> **Research log:** This notebook contains successive exploratory runs and revisions. It is preserved for provenance, not presented as a clean top-to-bottom reproduction. The original executed outputs have been cleared for release review.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

DATA_DIR = PROJECT / "data" / "processed"
RESULTS_DIR = PROJECT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "R1": pd.read_parquet(
        DATA_DIR / "development_R1_10k.parquet"
    ),
    "R2": pd.read_parquet(
        DATA_DIR / "development_R2_10k.parquet"
    ),
    "R3": pd.read_parquet(
        DATA_DIR / "development_R3_10k.parquet"
    )
}

for name, data in datasets.items():
    print(name, data.shape)

In [ ]:
DIAG_COLS = [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

CAT_COLS = [
    "age",
    *DIAG_COLS
]

NUM_COLS = {
    "R1": [],
    "R2": [
        "number_outpatient",
        "number_emergency",
        "number_inpatient"
    ],
    "R3": [
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "A1Cresult_observed",
        "max_glu_serum_observed"
    ]
}


def make_pipeline(name):

    categorical = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ])

    numeric = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            RobustScaler()
        )
    ])

    transformers = [
        ("cat", categorical, CAT_COLS)
    ]

    if NUM_COLS[name]:
        transformers.append(
            ("num", numeric, NUM_COLS[name])
        )

    return ColumnTransformer(transformers)


def fit_clusters(data, name, k=2):

    features = data.drop(
        columns=["patient_nbr"]
    )

    pipeline = make_pipeline(name)

    X = pipeline.fit_transform(features)

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    return model.fit_predict(X)

In [ ]:
baseline_labels = {}

for name, data in datasets.items():

    baseline_labels[name] = fit_clusters(
        data,
        name,
        k=2
    )

    print(
        name,
        "baseline cluster counts:",
        np.bincount(baseline_labels[name])
    )

print("\nSTEP 18C: PASS")

In [ ]:
def inject_missingness(data, rate, seed):

    rng = np.random.default_rng(seed)

    modified = data.copy()

    for col in DIAG_COLS:

        observed = (
            modified[col].notna()
            & (modified[col].astype(str) != "missing")
        )

        eligible = np.flatnonzero(
            observed.to_numpy()
        )

        n_mask = round(len(eligible) * rate)

        selected = rng.choice(
            eligible,
            size=n_mask,
            replace=False
        )

        modified.loc[selected, col] = "missing"

    return modified

In [ ]:
MISSING_RATES = [0.05, 0.10, 0.20, 0.30]
REPETITIONS = 5

rows = []

for name, data in datasets.items():

    for rate in MISSING_RATES:

        for repetition in range(REPETITIONS):

            seed = 1000 + repetition

            perturbed = inject_missingness(
                data,
                rate,
                seed
            )

            new_labels = fit_clusters(
                perturbed,
                name,
                k=2
            )

            ari = adjusted_rand_score(
                baseline_labels[name],
                new_labels
            )

            rows.append({
                "representation": name,
                "missing_rate": rate,
                "repetition": repetition,
                "ARI": ari
            })

        print(
            f"{name} | missing={rate:.0%} complete"
        )

missing_results = pd.DataFrame(rows)

print("\nExperiment rows:", len(missing_results))

assert len(missing_results) == 60

print("STEP 18E: PASS")

In [ ]:
missing_summary = (
    missing_results
    .groupby(["representation", "missing_rate"])
    .agg(
        mean_ARI=("ARI", "mean"),
        std_ARI=("ARI", "std"),
        min_ARI=("ARI", "min"),
        max_ARI=("ARI", "max")
    )
    .reset_index()
)

display(missing_summary.round(4))

missing_results.to_csv(
    RESULTS_DIR / "step18_missingness_runs.csv",
    index=False
)

missing_summary.to_csv(
    RESULTS_DIR / "step18_missingness_summary.csv",
    index=False
)

assert len(missing_summary) == 12

print("\nSTEP 18 VALIDATION: PASS")

In [ ]:
for name, data in datasets.items():

    labels = baseline_labels[name]

    counts = pd.Series(labels).value_counts().sort_index()

    print("\n", name)
    print("Cluster counts:")
    print(counts)

    print(
        "Smallest cluster percentage:",
        round(100 * counts.min() / len(labels), 2)
    )

In [ ]:
for name in ["R2", "R3"]:

    temp = datasets[name].copy()

    temp["cluster"] = baseline_labels[name]

    print("\n", name)

    display(
        temp.groupby("cluster")[
            [
                "number_outpatient",
                "number_emergency",
                "number_inpatient"
            ]
        ].agg(["mean", "median"]).round(2)
    )

In [ ]:
from sklearn.metrics import pairwise_distances

rng = np.random.default_rng(42)

for name in ["R1", "R2", "R3"]:

    data = datasets[name]

    prep = make_pipeline(name)

    X = prep.fit_transform(
        data.drop(columns=["patient_nbr"])
    )

    if hasattr(X, "toarray"):
        X = X.toarray()

    idx = rng.choice(
        len(X),
        size=500,
        replace=False
    )

    sample = X[idx]

    cat_width = len(
        prep.named_transformers_["cat"]
        .named_steps["encoder"]
        .get_feature_names_out()
    )

    categorical = sample[:, :cat_width]
    numeric = sample[:, cat_width:]

    cat_dist = pairwise_distances(
        categorical,
        metric="sqeuclidean"
    )

    cat_mean = cat_dist[
        np.triu_indices_from(cat_dist, k=1)
    ].mean()

    if numeric.shape[1] > 0:

        num_dist = pairwise_distances(
            numeric,
            metric="sqeuclidean"
        )

        num_mean = num_dist[
            np.triu_indices_from(num_dist, k=1)
        ].mean()

    else:
        num_mean = 0.0

    total = cat_mean + num_mean

    print("\n", name)
    print("Categorical distance:", round(cat_mean, 3))
    print("Numeric distance:", round(num_mean, 3))
    print(
        "Numeric contribution (%):",
        round(100 * num_mean / total, 2)
        if total > 0 else 0
    )

In [ ]:
from sklearn.metrics import adjusted_rand_score

ari_r2_r3 = adjusted_rand_score(
    baseline_labels["R2"],
    baseline_labels["R3"]
)

ari_r1_r2 = adjusted_rand_score(
    baseline_labels["R1"],
    baseline_labels["R2"]
)

print("R2 vs R3 ARI:", round(ari_r2_r3, 4))
print("R1 vs R2 ARI:", round(ari_r1_r2, 4))

print("\nSTEP 18G: COMPLETE")

In [ ]:
import pandas as pd
import numpy as np

for name in ["R1", "R2", "R3"]:
    counts = pd.Series(
        baseline_labels[name]
    ).value_counts().sort_index()

    print("\n", name)
    print(counts)
    print("Smallest cluster %:",
          round(100 * counts.min() / counts.sum(), 2))

In [ ]:
for name in ["R2", "R3"]:

    temp = datasets[name].copy()
    temp["cluster"] = baseline_labels[name]

    print("\n", name)

    display(
        temp.groupby("cluster")[
            [
                "number_outpatient",
                "number_emergency",
                "number_inpatient"
            ]
        ].agg(["mean", "median", "max"]).round(2)
    )

In [ ]:
for name in ["R1", "R2", "R3"]:

    temp = datasets[name].copy()
    temp["cluster"] = baseline_labels[name]

    print("\n", name, "Primary diagnosis composition")

    table = pd.crosstab(
        temp["cluster"],
        temp["diag1_family"],
        normalize="index"
    ) * 100

    display(table.round(1))

In [ ]:
for name in ["R1", "R2", "R3"]:

    temp = datasets[name].copy()
    temp["cluster"] = baseline_labels[name]

    print(f"\n===== {name} =====")

    sizes = temp["cluster"].value_counts().sort_index()
    display(
        pd.DataFrame({
            "patients": sizes,
            "percentage": (sizes / len(temp) * 100).round(2)
        })
    )

    if name in ["R2", "R3"]:
        display(
            temp.groupby("cluster")[
                [
                    "number_outpatient",
                    "number_emergency",
                    "number_inpatient"
                ]
            ].agg(["mean", "median", "max"]).round(2)
        )

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import adjusted_rand_score
import numpy as np
import pandas as pd

util_cols = [
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

X_util = datasets["R2"][util_cols].copy()

X_util = X_util.fillna(X_util.median())

scaler = RobustScaler()
X_util_scaled = scaler.fit_transform(X_util)

model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

util_labels = model.fit_predict(X_util_scaled)

print("Utilization-only cluster sizes:")
print(pd.Series(util_labels).value_counts())

print(
    "\nARI: utilization-only vs R2:",
    round(
        adjusted_rand_score(
            util_labels,
            baseline_labels["R2"]
        ),
        4
    )
)

print("\nSTEP 19A: PASS")

In [ ]:
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import numpy as np
import pandas as pd

data = datasets["R2"].copy()

clinical_cols = [
    "age",
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

util_cols = [
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

clinical_pipe = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="missing"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

util_pipe = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median"
    )),
    ("scaler", RobustScaler())
])

X_clin = clinical_pipe.fit_transform(
    data[clinical_cols]
)

X_util = util_pipe.fit_transform(
    data[util_cols]
)

# Average squared distance between independent patients
# equals 2 times the sum of feature variances.
clin_distance = 2 * np.var(
    X_clin, axis=0
).sum()

util_distance = 2 * np.var(
    X_util, axis=0
).sum()

print("Clinical distance:", round(clin_distance, 4))
print("Utilization distance:", round(util_distance, 4))

# Normalize each block to unit expected squared distance.
X_clin_bal = X_clin / np.sqrt(clin_distance)
X_util_bal = X_util / np.sqrt(util_distance)

X_balanced = np.hstack([
    X_clin_bal,
    X_util_bal
])

print("Balanced matrix:", X_balanced.shape)
print("STEP 19B-1: PASS")

In [ ]:
balanced_model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

balanced_labels = balanced_model.fit_predict(
    X_balanced
)

print("Balanced cluster sizes:")
print(
    pd.Series(balanced_labels)
    .value_counts()
    .sort_index()
)

print(
    "\nBalanced vs R1 ARI:",
    round(
        adjusted_rand_score(
            balanced_labels,
            baseline_labels["R1"]
        ),
        4
    )
)

print(
    "Balanced vs R2 ARI:",
    round(
        adjusted_rand_score(
            balanced_labels,
            baseline_labels["R2"]
        ),
        4
    )
)

print("STEP 19B-2: PASS")

In [ ]:
temp = data.copy()
temp["balanced_cluster"] = balanced_labels

print("Primary diagnosis distribution (%):")

display(
    (
        pd.crosstab(
            temp["balanced_cluster"],
            temp["diag1_family"],
            normalize="index"
        ) * 100
    ).round(1)
)

print("\nUtilization by balanced cluster:")

display(
    temp.groupby("balanced_cluster")[util_cols]
    .agg(["mean", "median"])
    .round(2)
)

print("STEP 19B-3: PASS")

In [ ]:
from sklearn.metrics import adjusted_rand_score
import pandas as pd

print("Balanced cluster sizes:")
print(
    pd.Series(balanced_labels)
    .value_counts()
    .sort_index()
)

print(
    "\nBalanced vs R1 ARI:",
    round(
        adjusted_rand_score(
            balanced_labels,
            baseline_labels["R1"]
        ), 4
    )
)

print(
    "Balanced vs R2 ARI:",
    round(
        adjusted_rand_score(
            balanced_labels,
            baseline_labels["R2"]
        ), 4
    )
)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import pandas as pd
import numpy as np

weights = [0.0, 0.25, 0.50, 0.75, 0.90, 1.0]

rows = []
weighted_labels = {}

for w in weights:

    X_weighted = np.hstack([
        np.sqrt(w) * X_clin_bal,
        np.sqrt(1 - w) * X_util_bal
    ])

    model = KMeans(
        n_clusters=2,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_weighted)

    weighted_labels[w] = labels

    counts = np.bincount(labels)

    rows.append({
        "clinical_weight": w,
        "utilization_weight": 1 - w,
        "smallest_cluster": counts.min(),
        "largest_cluster": counts.max(),
        "ARI_vs_R1": adjusted_rand_score(
            labels, baseline_labels["R1"]
        ),
        "ARI_vs_R2": adjusted_rand_score(
            labels, baseline_labels["R2"]
        )
    })

weight_summary = pd.DataFrame(rows)

display(weight_summary.round(4))

weight_summary.to_csv(
    RESULTS_DIR / "step19c_weight_sensitivity.csv",
    index=False
)

print("STEP 19C-1: PASS")

In [ ]:
comparison = []

for i in range(len(weights) - 1):

    w1 = weights[i]
    w2 = weights[i + 1]

    comparison.append({
        "from_weight": w1,
        "to_weight": w2,
        "ARI_between_weights": adjusted_rand_score(
            weighted_labels[w1],
            weighted_labels[w2]
        )
    })

transition_summary = pd.DataFrame(comparison)

display(transition_summary.round(4))

transition_summary.to_csv(
    RESULTS_DIR / "step19c_weight_transitions.csv",
    index=False
)

print("STEP 19C-2: PASS")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    weight_summary["clinical_weight"],
    weight_summary["ARI_vs_R1"],
    marker="o",
    label="Agreement with clinical-only R1"
)

plt.plot(
    weight_summary["clinical_weight"],
    weight_summary["ARI_vs_R2"],
    marker="s",
    label="Agreement with utilization-oriented R2"
)

plt.xlabel("Clinical block weight")
plt.ylabel("Adjusted Rand Index")
plt.title("Sensitivity of Patient Clusters to Representation Weighting")
plt.ylim(-0.1, 1.05)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import numpy as np
import pandas as pd

fine_weights = np.round(
    np.arange(0.75, 0.951, 0.025), 3
)

seeds = range(10)

rows = []

for w in fine_weights:

    X_weighted = np.hstack([
        np.sqrt(w) * X_clin_bal,
        np.sqrt(1 - w) * X_util_bal
    ])

    for seed in seeds:

        model = KMeans(
            n_clusters=2,
            random_state=seed,
            n_init=20
        )

        labels = model.fit_predict(X_weighted)

        counts = np.bincount(labels)

        rows.append({
            "clinical_weight": w,
            "seed": seed,
            "ARI_vs_R1": adjusted_rand_score(
                labels, baseline_labels["R1"]
            ),
            "ARI_vs_R2": adjusted_rand_score(
                labels, baseline_labels["R2"]
            ),
            "smallest_cluster": counts.min()
        })

fine_results = pd.DataFrame(rows)

print("Experiment runs:", len(fine_results))

assert len(fine_results) == 90

print("STEP 19D-1: PASS")

In [ ]:
fine_summary = (
    fine_results
    .groupby("clinical_weight")
    .agg(
        mean_ARI_R1=("ARI_vs_R1", "mean"),
        min_ARI_R1=("ARI_vs_R1", "min"),
        max_ARI_R1=("ARI_vs_R1", "max"),
        mean_ARI_R2=("ARI_vs_R2", "mean"),
        min_ARI_R2=("ARI_vs_R2", "min"),
        max_ARI_R2=("ARI_vs_R2", "max"),
        min_cluster_size=("smallest_cluster", "min"),
        max_cluster_size=("smallest_cluster", "max")
    )
    .reset_index()
)

display(fine_summary.round(4))

fine_results.to_csv(
    RESULTS_DIR / "step19d_fine_weight_runs.csv",
    index=False
)

fine_summary.to_csv(
    RESULTS_DIR / "step19d_fine_weight_summary.csv",
    index=False
)

print("STEP 19D-2: PASS")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    fine_summary["clinical_weight"],
    fine_summary["mean_ARI_R1"],
    marker="o",
    label="Agreement with clinical-only"
)

plt.plot(
    fine_summary["clinical_weight"],
    fine_summary["mean_ARI_R2"],
    marker="s",
    label="Agreement with utilization-oriented"
)

plt.xlabel("Clinical block weight")
plt.ylabel("Mean Adjusted Rand Index")
plt.title("Fine-Grained Representation Sensitivity")
plt.ylim(-0.1, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
transition_check = (
    fine_results
    .assign(
        matches_clinical=lambda x: x["ARI_vs_R1"] > 0.95,
        matches_utilization=lambda x: x["ARI_vs_R2"] > 0.95
    )
    .groupby("clinical_weight")
    .agg(
        runs=("seed", "count"),
        clinical_matches=("matches_clinical", "sum"),
        utilization_matches=("matches_utilization", "sum"),
        min_cluster=("smallest_cluster", "min"),
        max_cluster=("smallest_cluster", "max")
    )
    .reset_index()
)

display(transition_check)

print("\nSTEP 19E: PASS")

In [ ]:
from pathlib import Path

RESULTS_DIR = Path(
    r"C:\Projects\healthcare-cluster-reliability\results"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

transition_check.to_csv(
    RESULTS_DIR / "step19e_transition_reproducibility.csv",
    index=False
)

assert len(transition_check) == 9
assert transition_check["runs"].eq(10).all()

print("STEP 19E SAVED: PASS")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import adjusted_rand_score

DIAG_COLS = [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

MISSING_RATES = [0.05, 0.10, 0.20, 0.30]
REPETITIONS = 10

data = datasets["R2"].copy()

utilization = (
    data["number_outpatient"].fillna(0)
    + data["number_emergency"].fillna(0)
    + data["number_inpatient"].fillna(0)
)

high_utilization = (
    utilization >= utilization.quantile(0.75)
)

print("Patients:", len(data))
print("High-utilization patients:", high_utilization.sum())
print("Step 20A: PASS")

In [ ]:
print("Utilization quantiles:")
print(utilization.quantile([0.50, 0.75, 0.90, 0.95]))

print("\nHigh-utilization percentage:")
print(round(high_utilization.mean() * 100, 2))

In [ ]:
import numpy as np
import pandas as pd

DIAG_COLS = [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

def structured_missingness(data, rate, mechanism, seed):

    rng = np.random.default_rng(seed)
    modified = data.copy()

    utilization = (
        data["number_outpatient"].fillna(0)
        + data["number_emergency"].fillna(0)
        + data["number_inpatient"].fillna(0)
    )

    any_utilization = (utilization >= 1).to_numpy()

    for col in DIAG_COLS:

        observed = (
            data[col].notna()
            & data[col].astype(str).ne("missing")
        ).to_numpy()

        eligible = np.flatnonzero(observed)
        n_mask = round(rate * len(eligible))

        if mechanism == "MCAR":
            weights = np.ones(len(eligible))

        elif mechanism == "UTIL_ASSOC":
            weights = np.where(
                any_utilization[eligible],
                3.0,
                1.0
            )

        else:
            raise ValueError("Unknown mechanism")

        weights = weights / weights.sum()

        selected = rng.choice(
            eligible,
            size=n_mask,
            replace=False,
            p=weights
        )

        modified.iloc[
            selected,
            modified.columns.get_loc(col)
        ] = "missing"

    return modified

In [ ]:
rate = 0.20

for mechanism in ["MCAR", "UTIL_ASSOC"]:

    test = structured_missingness(
        datasets["R2"],
        rate=rate,
        mechanism=mechanism,
        seed=42
    )

    original = datasets["R2"]

    changed = pd.DataFrame({
        col: (
            original[col].astype(str).to_numpy()
            != test[col].astype(str).to_numpy()
        )
        for col in DIAG_COLS
    })

    utilization = (
        original["number_outpatient"].fillna(0)
        + original["number_emergency"].fillna(0)
        + original["number_inpatient"].fillna(0)
    )

    groups = np.where(
        utilization.to_numpy() >= 1,
        "Any utilization",
        "No utilization"
    )

    print("\n", mechanism)
    print("Overall masked fraction:",
          round(changed.to_numpy().mean(), 4))

    print(
        changed.assign(group=groups)
        .groupby("group")
        .mean()
        .round(4)
    )

print("\nSTEP 20C: PASS")

In [ ]:
import numpy as np
import pandas as pd

DIAG_COLS = [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

UTIL_COLS = [
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

# Reference utilization information from R2
util_reference = (
    datasets["R2"]
    .set_index("patient_nbr")[UTIL_COLS]
    .fillna(0)
)

assert util_reference.index.is_unique


def structured_missingness(data, rate, mechanism, seed):

    rng = np.random.default_rng(seed)
    modified = data.copy()

    # Match utilization to each patient, including R1 patients
    util = util_reference.reindex(
        data["patient_nbr"].to_numpy()
    )

    assert not util.isna().any().any(), (
        "Some patient IDs are missing from R2"
    )

    any_utilization = (
        util.sum(axis=1).to_numpy() >= 1
    )

    for col in DIAG_COLS:

        observed = (
            data[col].notna()
            & data[col].astype(str).ne("missing")
        ).to_numpy()

        eligible = np.flatnonzero(observed)

        n_mask = round(rate * len(eligible))

        if mechanism == "MCAR":
            weights = np.ones(len(eligible))

        elif mechanism == "UTIL_ASSOC":
            weights = np.where(
                any_utilization[eligible],
                3.0,
                1.0
            )

        else:
            raise ValueError(
                f"Unknown mechanism: {mechanism}"
            )

        weights = weights / weights.sum()

        selected = rng.choice(
            eligible,
            size=n_mask,
            replace=False,
            p=weights
        )

        modified.iloc[
            selected,
            modified.columns.get_loc(col)
        ] = "missing"

    return modified


print("STEP 20C FUNCTION FIXED: PASS")

In [ ]:
for name in ["R1", "R2"]:

    test = structured_missingness(
        datasets[name],
        rate=0.20,
        mechanism="UTIL_ASSOC",
        seed=42
    )

    print(
        name,
        "Original:", datasets[name].shape,
        "Perturbed:", test.shape
    )

    assert test.shape == datasets[name].shape

print("\nSTEP 20 FIX VALIDATION: PASS")

In [ ]:
from sklearn.metrics import adjusted_rand_score

MISSING_RATES = [0.05, 0.10, 0.20, 0.30]
REPETITIONS = 10

rows = []

for name in ["R1", "R2"]:

    original = datasets[name]
    reference = baseline_labels[name]

    for mechanism in ["MCAR", "UTIL_ASSOC"]:

        for rate in MISSING_RATES:

            for rep in range(REPETITIONS):

                perturbed = structured_missingness(
                    original,
                    rate=rate,
                    mechanism=mechanism,
                    seed=2000 + rep
                )

                labels = fit_clusters(
                    perturbed,
                    name,
                    k=2
                )

                ari = adjusted_rand_score(
                    reference,
                    labels
                )

                rows.append({
                    "representation": name,
                    "mechanism": mechanism,
                    "missing_rate": rate,
                    "repetition": rep,
                    "ARI": ari,
                    "smallest_cluster": np.bincount(labels).min()
                })

            print(name, mechanism, rate, "complete")

step20_results = pd.DataFrame(rows)

assert len(step20_results) == 160

print("\nSTEP 20D: PASS")

In [ ]:
step20_summary = (
    step20_results
    .groupby([
        "representation",
        "mechanism",
        "missing_rate"
    ])
    .agg(
        mean_ARI=("ARI", "mean"),
        std_ARI=("ARI", "std"),
        min_ARI=("ARI", "min"),
        max_ARI=("ARI", "max"),
        min_cluster=("smallest_cluster", "min")
    )
    .reset_index()
)

display(step20_summary.round(4))

step20_results.to_csv(
    RESULTS_DIR / "step20_structured_missingness_runs.csv",
    index=False
)

step20_summary.to_csv(
    RESULTS_DIR / "step20_structured_missingness_summary.csv",
    index=False
)

assert len(step20_summary) == 16

print("\nSTEP 20E: PASS")

In [ ]:
    utilization_source = datasets["R2"].set_index("patient_nbr")

    utilization = (
        utilization_source.loc[
            data["patient_nbr"], "number_outpatient"
        ].fillna(0).to_numpy()
        + utilization_source.loc[
            data["patient_nbr"], "number_emergency"
        ].fillna(0).to_numpy()
        + utilization_source.loc[
            data["patient_nbr"], "number_inpatient"
        ].fillna(0).to_numpy()
    )

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

DATA_DIR = PROJECT / "data" / "processed"
RESULTS_DIR = PROJECT / "results"

data = pd.read_parquet(
    DATA_DIR / "development_R1_10k.parquet"
)

clinical_cols = [
    "age",
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

preprocessor = ColumnTransformer([
    (
        "clinical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="missing"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        clinical_cols
    )
])

X_reference = preprocessor.fit_transform(
    data[clinical_cols]
)

model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

reference_labels = model.fit_predict(
    X_reference
)

print("Patients:", len(data))
print(
    "Reference cluster sizes:",
    np.bincount(reference_labels)
)

print("STEP 21A: PASS")

In [ ]:
DIAG_COLS = [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

N_REPETITIONS = 20
MISSING_RATE = 0.20

assignment_changes = np.zeros(
    len(data),
    dtype=int
)

for rep in range(N_REPETITIONS):

    rng = np.random.default_rng(5000 + rep)

    perturbed = data.copy()

    for col in DIAG_COLS:

        observed = (
            perturbed[col].notna()
            & perturbed[col].astype(str).ne("missing")
        ).to_numpy()

        eligible = np.flatnonzero(observed)

        n_mask = round(
            MISSING_RATE * len(eligible)
        )

        selected = rng.choice(
            eligible,
            size=n_mask,
            replace=False
        )

        perturbed.iloc[
            selected,
            perturbed.columns.get_loc(col)
        ] = "missing"

    X_perturbed = preprocessor.transform(
        perturbed[clinical_cols]
    )

    predicted_labels = model.predict(
        X_perturbed
    )

    assignment_changes += (
        predicted_labels != reference_labels
    ).astype(int)

    print(
        f"Repetition {rep + 1}/{N_REPETITIONS} complete"
    )

print("STEP 21B: PASS")

In [ ]:
patient_uncertainty = pd.DataFrame({
    "patient_nbr": data["patient_nbr"].to_numpy(),
    "reference_cluster": reference_labels,
    "assignment_changes": assignment_changes,
    "change_fraction": (
        assignment_changes / N_REPETITIONS
    )
})

patient_uncertainty["assignment_stability"] = (
    1 - patient_uncertainty["change_fraction"]
)

print("Patient assignment stability summary:")

display(
    patient_uncertainty[
        "assignment_stability"
    ].describe().round(4)
)

print("\nPatients with assignment changes:")

print(
    (
        patient_uncertainty[
            "assignment_changes"
        ] > 0
    ).sum()
)

print("\nPatients changing in at least half the runs:")

print(
    (
        patient_uncertainty[
            "change_fraction"
        ] >= 0.50
    ).sum()
)

patient_uncertainty.to_csv(
    RESULTS_DIR / "step21_patient_uncertainty.csv",
    index=False
)

print("\nSTEP 21C: PASS")

In [ ]:
cluster_stability = (
    patient_uncertainty
    .groupby("reference_cluster")
    .agg(
        patients=("patient_nbr", "count"),
        mean_stability=("assignment_stability", "mean"),
        median_stability=("assignment_stability", "median"),
        patients_changed=(
            "assignment_changes",
            lambda x: (x > 0).sum()
        ),
        frequent_switchers=(
            "change_fraction",
            lambda x: (x >= 0.50).sum()
        )
    )
    .reset_index()
)

cluster_stability["changed_pct"] = (
    100 * cluster_stability["patients_changed"]
    / cluster_stability["patients"]
)

display(cluster_stability.round(4))

In [ ]:
analysis = patient_uncertainty.merge(
    data[["patient_nbr", "diag1_family"]],
    on="patient_nbr",
    how="left",
    validate="one_to_one"
)

diagnosis_stability = (
    analysis
    .groupby("diag1_family")
    .agg(
        patients=("patient_nbr", "count"),
        mean_stability=("assignment_stability", "mean"),
        changed_patients=(
            "assignment_changes",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)

diagnosis_stability["changed_pct"] = (
    100 * diagnosis_stability["changed_patients"]
    / diagnosis_stability["patients"]
)

display(
    diagnosis_stability
    .sort_values("changed_pct", ascending=False)
    .round(3)
)

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cluster_stability.to_csv(
    RESULTS_DIR / "step21d_cluster_stability.csv",
    index=False
)

diagnosis_stability.to_csv(
    RESULTS_DIR / "step21d_diagnosis_stability.csv",
    index=False
)

assert cluster_stability["patients"].sum() == 10000
assert diagnosis_stability["patients"].sum() == 10000

print("STEP 21D: PASS")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

DATA_DIR = PROJECT / "data" / "processed"
RESULTS_DIR = PROJECT / "results"

cohort = pd.read_parquet(
    DATA_DIR / "uci_diabetes_index_patient_features_v2.parquet"
)

print("Cohort shape:", cohort.shape)
print("Unique patients:", cohort["patient_nbr"].nunique())

assert cohort["patient_nbr"].is_unique

print("STEP 22A: PASS")

In [ ]:
dev = pd.read_parquet(
    DATA_DIR / "development_R1_10k.parquet"
)

profile = dev[["patient_nbr"]].merge(
    cohort,
    on="patient_nbr",
    how="left",
    validate="one_to_one"
)

assert len(profile) == 10000

print("Development profile shape:", profile.shape)
print("STEP 22B: PASS")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score

R1 = pd.read_parquet(
    DATA_DIR / "development_R1_10k.parquet"
)

R2 = pd.read_parquet(
    DATA_DIR / "development_R2_10k.parquet"
)

assert R1["patient_nbr"].is_unique
assert R2["patient_nbr"].is_unique

assert set(R1["patient_nbr"]) == set(R2["patient_nbr"])

print("R1:", R1.shape)
print("R2:", R2.shape)
print("STEP 22C-1: PASS")

In [ ]:
CAT_COLS = [
    "age",
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

UTIL_COLS = [
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

def build_model(representation):

    transformers = [
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="missing"
                    )
                ),
                (
                    "encoder",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            CAT_COLS
        )
    ]

    if representation == "R2":
        transformers.append(
            (
                "num",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(strategy="median")
                    ),
                    (
                        "scaler",
                        RobustScaler()
                    )
                ]),
                UTIL_COLS
            )
        )

    preprocessor = ColumnTransformer(transformers)

    model = Pipeline([
        ("preprocessor", preprocessor),
        (
            "kmeans",
            KMeans(
                n_clusters=2,
                random_state=42,
                n_init=20
            )
        )
    ])

    return model

In [ ]:
model_R1 = build_model("R1")
model_R2 = build_model("R2")

model_R1.fit(R1)
model_R2.fit(R2)

labels_R1 = model_R1.named_steps[
    "kmeans"
].labels_

labels_R2 = model_R2.named_steps[
    "kmeans"
].labels_

assignments_R1 = pd.DataFrame({
    "patient_nbr": R1["patient_nbr"].to_numpy(),
    "cluster_R1": labels_R1
})

assignments_R2 = pd.DataFrame({
    "patient_nbr": R2["patient_nbr"].to_numpy(),
    "cluster_R2": labels_R2
})

profile = (
    profile
    .merge(
        assignments_R1,
        on="patient_nbr",
        validate="one_to_one"
    )
    .merge(
        assignments_R2,
        on="patient_nbr",
        validate="one_to_one"
    )
)

assert profile["cluster_R1"].notna().all()
assert profile["cluster_R2"].notna().all()

print("Profile shape:", profile.shape)

print("\nR1 cluster sizes:")
print(profile["cluster_R1"].value_counts())

print("\nR2 cluster sizes:")
print(profile["cluster_R2"].value_counts())

print(
    "\nR1 vs R2 ARI:",
    round(
        adjusted_rand_score(
            profile["cluster_R1"],
            profile["cluster_R2"]
        ),
        4
    )
)

print("\nSTEP 22C-3: PASS")

In [ ]:
def phenotype_summary(df, cluster_col):

    summary = (
        df.groupby(cluster_col)
        .agg(
            patients=("patient_nbr", "count"),
            mean_outpatient=("number_outpatient", "mean"),
            mean_emergency=("number_emergency", "mean"),
            mean_inpatient=("number_inpatient", "mean"),
            mean_diagnoses=("number_diagnoses", "mean"),
            mean_hospital_days=("time_in_hospital", "mean"),
            mean_medications=("num_medications", "mean")
        )
        .round(3)
    )

    summary["percentage"] = (
        100 * summary["patients"] / len(df)
    ).round(2)

    return summary


print("R1 clinical phenotype profile:")
display(
    phenotype_summary(profile, "cluster_R1")
)

print("\nR2 utilization phenotype profile:")
display(
    phenotype_summary(profile, "cluster_R2")
)

In [ ]:
profile.to_parquet(
    RESULTS_DIR / "step22_patient_cluster_profiles.parquet",
    index=False
)

phenotype_summary(
    profile, "cluster_R1"
).to_csv(
    RESULTS_DIR / "step22_R1_phenotype_summary.csv"
)

phenotype_summary(
    profile, "cluster_R2"
).to_csv(
    RESULTS_DIR / "step22_R2_phenotype_summary.csv"
)

print("STEP 22E: PASS")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

RESULTS_DIR = PROJECT / "results"

profile = pd.read_parquet(
    RESULTS_DIR / "step22_patient_cluster_profiles.parquet"
)

print("Dataset shape:", profile.shape)
print("Unique patients:", profile["patient_nbr"].nunique())

required = [
    "patient_nbr",
    "cluster_R1",
    "cluster_R2",
    "readmitted",
    "readmit_30d"
]

assert all(col in profile.columns for col in required)
assert profile["patient_nbr"].is_unique
assert len(profile) == 10000

print("\nReadmission outcome distribution:")
print(profile["readmitted"].value_counts(dropna=False))

print("\nSTEP 23A: PASS")

In [ ]:
def readmission_summary(df, cluster_col):

    result = (
        df.groupby(cluster_col)
        .agg(
            patients=("patient_nbr", "count"),
            readmissions=("readmit_30d", "sum"),
            readmission_rate=("readmit_30d", "mean")
        )
        .reset_index()
    )

    result["readmission_pct"] = (
        result["readmission_rate"] * 100
    ).round(2)

    return result


r1_outcomes = readmission_summary(
    profile, "cluster_R1"
)

r2_outcomes = readmission_summary(
    profile, "cluster_R2"
)

print("R1: Clinical-oriented clusters")
display(r1_outcomes)

print("\nR2: Utilization-oriented clusters")
display(r2_outcomes)

print("\nSTEP 23B: PASS")

In [ ]:
from scipy.stats import chi2_contingency

for cluster_col in ["cluster_R1", "cluster_R2"]:

    contingency = pd.crosstab(
        profile[cluster_col],
        profile["readmit_30d"]
    )

    chi2, p_value, dof, expected = chi2_contingency(
        contingency,
        correction=False
    )

    print(f"\n{cluster_col}")
    print(contingency)

    print("Chi-square:", round(chi2, 4))
    print("P-value:", p_value)

    print(
        "Minimum expected cell count:",
        round(expected.min(), 2)
    )

print("\nSTEP 23C: PASS")

In [ ]:
for cluster_col in ["cluster_R1", "cluster_R2"]:

    rates = (
        profile.groupby(cluster_col)["readmit_30d"]
        .mean()
        .sort_index()
    )

    difference = (
        rates.max() - rates.min()
    ) * 100

    print(
        cluster_col,
        "Absolute readmission rate difference:",
        round(difference, 2),
        "percentage points"
    )

print("\nSTEP 23D: PASS")

In [ ]:
r1_outcomes.to_csv(
    RESULTS_DIR / "step23_R1_readmission.csv",
    index=False
)

r2_outcomes.to_csv(
    RESULTS_DIR / "step23_R2_readmission.csv",
    index=False
)

print("STEP 23E: PASS")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    adjusted_rand_score,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")
DATA = PROJECT / "data" / "processed"
RESULTS = PROJECT / "results"
RESULTS.mkdir(exist_ok=True, parents=True)

clinical_cols = ["age", "diag1_family", "diag2_family", "diag3_family"]
util_cols = ["number_outpatient", "number_emergency", "number_inpatient"]

datasets = {
    "R1": pd.read_parquet(DATA / "development_R1_10k.parquet"),
    "R2": pd.read_parquet(DATA / "development_R2_10k.parquet")
}

def preprocess(df, representation):
    transformers = [
        ("clinical", Pipeline([
            ("imputer", SimpleImputer(
                strategy="constant", fill_value="missing"
            )),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), clinical_cols)
    ]

    if representation == "R2":
        transformers.append(
            ("util", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scale", RobustScaler())
            ]), util_cols)
        )

    prep = ColumnTransformer(transformers)
    X = prep.fit_transform(df)
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)

X_data = {
    name: preprocess(df, name)
    for name, df in datasets.items()
}

print({name: X.shape for name, X in X_data.items()})
print("STEP 24A: PASS")

In [ ]:
K_VALUES = [2, 3, 5, 8]
SEEDS = [0, 1, 2, 3, 4]

rows = []

for representation, X in X_data.items():
    for algorithm in ["KMeans", "MiniBatchKMeans", "Agglomerative"]:
        for k in K_VALUES:

            # Agglomerative is evaluated on a fixed 3,000-patient subset.
            if algorithm == "Agglomerative":
                idx = np.random.default_rng(42).choice(
                    len(X), size=3000, replace=False
                )
                idx.sort()
                X_eval = X[idx]
                seeds_to_run = [42]
            else:
                X_eval = X
                seeds_to_run = SEEDS

            for seed in seeds_to_run:
                if algorithm == "KMeans":
                    model = KMeans(
                        n_clusters=k, n_init=20, random_state=seed
                    )
                elif algorithm == "MiniBatchKMeans":
                    model = MiniBatchKMeans(
                        n_clusters=k, n_init=10,
                        batch_size=1024, random_state=seed
                    )
                else:
                    model = AgglomerativeClustering(
                        n_clusters=k, linkage="ward"
                    )

                labels = model.fit_predict(X_eval)

                counts = np.bincount(labels)

                if counts.min() < 2:
                    sil = np.nan
                else:
                    sil = silhouette_score(
                        X_eval, labels,
                        sample_size=min(2000, len(X_eval)),
                        random_state=42
                    )

                rows.append({
                    "representation": representation,
                    "algorithm": algorithm,
                    "k": k,
                    "seed": seed,
                    "n_patients": len(X_eval),
                    "silhouette": sil,
                    "CH": calinski_harabasz_score(X_eval, labels),
                    "DB": davies_bouldin_score(X_eval, labels),
                    "min_cluster": counts.min(),
                    "max_cluster": counts.max()
                })

comparison = pd.DataFrame(rows)

comparison.to_csv(
    RESULTS / "step24_algorithm_comparison.csv", index=False
)

display(comparison.groupby(
    ["representation", "algorithm", "k"]
).agg(
    mean_silhouette=("silhouette", "mean"),
    min_cluster=("min_cluster", "min"),
    max_cluster=("max_cluster", "max"),
    runs=("seed", "count")
).round(4))

print("STEP 24B: PASS")

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")
RESULTS = PROJECT / "results"

seed = pd.read_csv(RESULTS / "step16_seed_stability.csv")
subsample = pd.read_csv(
    RESULTS / "step17_subsample_stability_summary.csv"
)
missing = pd.read_csv(
    RESULTS / "step20_structured_missingness_summary.csv"
)

print("Seed columns:", seed.columns.tolist())
print("Subsample columns:", subsample.columns.tolist())
print("Missingness columns:", missing.columns.tolist())

display(seed.head())
display(subsample.head())
display(missing.head())

print("STEP 25A: PASS")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")
DATA = PROJECT / "data" / "processed"
RESULTS = PROJECT / "results"

cohort = pd.read_parquet(
    DATA / "uci_diabetes_index_patient_features_v2.parquet"
)
dev = pd.read_parquet(DATA / "development_R1_10k.parquet")

heldout = cohort.loc[
    ~cohort["patient_nbr"].isin(dev["patient_nbr"])
].copy()

assert len(heldout) == 61518
assert heldout["patient_nbr"].is_unique
assert not set(heldout["patient_nbr"]) & set(dev["patient_nbr"])

print("Development patients:", len(dev))
print("Held-out patients:", len(heldout))
print("STEP 26A: PASS")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

CLIN = ["age", "diag1_family", "diag2_family", "diag3_family"]
UTIL = ["number_outpatient", "number_emergency", "number_inpatient"]

def make_preprocessor(rep):
    blocks = [
        ("clinical", Pipeline([
            ("imputer", SimpleImputer(
                strategy="constant", fill_value="missing"
            )),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), CLIN)
    ]

    if rep == "R2":
        blocks.append(
            ("util", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler())
            ]), UTIL)
        )

    return ColumnTransformer(blocks)

heldout_assignments = heldout[["patient_nbr", "readmit_30d"]].copy()

for rep in ["R1", "R2"]:
    train = pd.read_parquet(
        DATA / f"development_{rep}_10k.parquet"
    )

    prep = make_preprocessor(rep)
    X_train = prep.fit_transform(train)
    X_test = prep.transform(heldout)

    model = KMeans(
        n_clusters=2, random_state=42, n_init=20
    )
    model.fit(X_train)

    heldout_assignments[f"cluster_{rep}"] = model.predict(X_test)

    print(rep, "held-out cluster counts:")
    print(heldout_assignments[f"cluster_{rep}"].value_counts())

heldout_assignments.to_parquet(
    RESULTS / "step26_heldout_assignments.parquet",
    index=False
)

print("STEP 26B: PASS")

In [ ]:
for rep in ["R1", "R2"]:
    print(f"\n{rep} held-out readmission rates")

    display(
        heldout_assignments.groupby(f"cluster_{rep}")
        ["readmit_30d"]
        .agg(["count", "sum", "mean"])
        .round(4)
    )

print("STEP 26C: PASS")

In [ ]:
from pathlib import Path

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")
MIMIC = PROJECT / "data" / "raw" / "mimic"

files = sorted(
    p for p in MIMIC.rglob("*")
    if p.is_file()
)

print("MIMIC files found:", len(files))

for path in files[:60]:
    print(path.relative_to(MIMIC))

print("STEP 27A: COMPLETE")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import chi2_contingency

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")
RESULTS = PROJECT / "results"

df = pd.read_parquet(
    RESULTS / "step26_heldout_assignments.parquet"
)

rows = []

for rep in ["R1", "R2"]:
    cluster_col = f"cluster_{rep}"

    groups = (
        df.groupby(cluster_col)["readmit_30d"]
        .agg(["sum", "count", "mean"])
        .sort_index()
    )

    assert len(groups) == 2

    p0, p1 = groups["mean"].to_numpy()
    n0, n1 = groups["count"].to_numpy()

    risk_difference = p1 - p0

    standard_error = np.sqrt(
        p0 * (1 - p0) / n0
        + p1 * (1 - p1) / n1
    )

    lower = risk_difference - 1.96 * standard_error
    upper = risk_difference + 1.96 * standard_error

    contingency = pd.crosstab(
        df[cluster_col], df["readmit_30d"]
    )

    chi2, p_value, _, expected = chi2_contingency(
        contingency, correction=False
    )

    rows.append({
        "representation": rep,
        "cluster_0_rate": p0,
        "cluster_1_rate": p1,
        "risk_difference": risk_difference,
        "ci_lower": lower,
        "ci_upper": upper,
        "chi2_p": p_value,
        "minimum_expected": expected.min()
    })

statistics = pd.DataFrame(rows)

display(statistics.round(5))

statistics.to_csv(
    RESULTS / "step28_heldout_readmission_statistics.csv",
    index=False
)

print("STEP 28: PASS")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")
RESULTS = PROJECT / "results"
FIGURES = PROJECT / "figures"
FIGURES.mkdir(exist_ok=True, parents=True)

df = pd.read_csv(
    RESULTS / "step21_patient_uncertainty.csv"
)

plt.figure(figsize=(8, 5))

plt.hist(
    df["assignment_stability"],
    bins=20,
    edgecolor="black"
)

plt.xlabel("Patient assignment stability")
plt.ylabel("Number of patients")
plt.title("Patient-Level Assignment Robustness Under 20% Diagnosis Masking")

plt.tight_layout()

plt.savefig(
    FIGURES / "figure5_patient_assignment_stability.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("STEP 29: FIGURE SAVED")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

RESULTS = PROJECT / "results"

expected_files = {
    "Step 24 Algorithm comparison":
        "step24_algorithm_comparison.csv",

    "Step 25 Seed stability":
        "step16_seed_stability.csv",

    "Step 25 Subsampling stability":
        "step17_subsample_stability_summary.csv",

    "Step 25 Missingness":
        "step20_structured_missingness_summary.csv",

    "Step 26 Held-out validation":
        "step26_heldout_assignments.parquet",

    "Step 28 Statistical analysis":
        "step28_heldout_readmission_statistics.csv"
}

rows = []

for step, filename in expected_files.items():

    path = RESULTS / filename

    rows.append({
        "experiment": step,
        "file": filename,
        "exists": path.exists(),
        "size_kb": (
            round(path.stat().st_size / 1024, 2)
            if path.exists() else None
        )
    })

status = pd.DataFrame(rows)

display(status)

print("\nEXPERIMENT INVENTORY: PASS")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

RESULTS = PROJECT / "results"

expected_files = {
    "Step 24 Algorithm comparison":
        "step24_algorithm_comparison.csv",

    "Step 25 Seed stability":
        "step16_seed_stability.csv",

    "Step 25 Subsampling stability":
        "step17_subsample_stability_summary.csv",

    "Step 25 Missingness":
        "step20_structured_missingness_summary.csv",

    "Step 26 Held-out validation":
        "step26_heldout_assignments.parquet",

    "Step 28 Statistical analysis":
        "step28_heldout_readmission_statistics.csv"
}

rows = []

for step, filename in expected_files.items():

    path = RESULTS / filename

    rows.append({
        "experiment": step,
        "file": filename,
        "exists": path.exists(),
        "size_kb": (
            round(path.stat().st_size / 1024, 2)
            if path.exists() else None
        )
    })

status = pd.DataFrame(rows)

display(status)

print("\nEXPERIMENT INVENTORY: PASS")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

stability = pd.read_csv(
    RESULTS / "step21_patient_uncertainty.csv"
)

counts = (
    stability["assignment_stability"]
    .round(2)
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    counts.index,
    counts.values,
    width=0.035,
    edgecolor="black"
)

ax.set_xlabel("Assignment stability across 20 masking runs")
ax.set_ylabel("Number of patients")
ax.set_title("Patient-Level Assignment Robustness")

ax.set_xticks(np.arange(0.45, 1.01, 0.05))
ax.tick_params(axis="x", rotation=45)

ax.text(
    0.02, 0.97,
    "N = 10,000 | Mean stability = 93.84%\n"
    "20% diagnosis masking; fixed R1 model",
    transform=ax.transAxes,
    va="top",
    fontsize=10
)

fig.tight_layout()

fig.savefig(
    PROJECT / "figures" /
    "figure5_patient_stability_final.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("FIGURE 5 FINAL: PASS")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

RESULTS = PROJECT / "results"

expected_files = {
    "Step 24 Algorithm comparison":
        "step24_algorithm_comparison.csv",

    "Step 25 Seed stability":
        "step16_seed_stability.csv",

    "Step 25 Subsampling stability":
        "step17_subsample_stability_summary.csv",

    "Step 25 Missingness":
        "step20_structured_missingness_summary.csv",

    "Step 26 Held-out validation":
        "step26_heldout_assignments.parquet",

    "Step 28 Statistical analysis":
        "step28_heldout_readmission_statistics.csv"
}

rows = []

for step, filename in expected_files.items():

    path = RESULTS / filename

    rows.append({
        "experiment": step,
        "file": filename,
        "exists": path.exists(),
        "size_kb": (
            round(path.stat().st_size / 1024, 2)
            if path.exists() else None
        )
    })

status = pd.DataFrame(rows)

display(status)

print("\nEXPERIMENT INVENTORY: PASS")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

stability = pd.read_csv(
    RESULTS / "step21_patient_uncertainty.csv"
)

counts = (
    stability["assignment_stability"]
    .round(2)
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    counts.index,
    counts.values,
    width=0.035,
    edgecolor="black"
)

ax.set_xlabel("Assignment stability across 20 masking runs")
ax.set_ylabel("Number of patients")
ax.set_title("Patient-Level Assignment Robustness")

ax.set_xticks(np.arange(0.45, 1.01, 0.05))
ax.tick_params(axis="x", rotation=45)

ax.text(
    0.02, 0.97,
    "N = 10,000 | Mean stability = 93.84%\n"
    "20% diagnosis masking; fixed R1 model",
    transform=ax.transAxes,
    va="top",
    fontsize=10
)

fig.tight_layout()

fig.savefig(
    PROJECT / "figures" /
    "figure5_patient_stability_final.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("FIGURE 5 FINAL: PASS")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

RESULTS = PROJECT / "results"

expected = {
    "24 Algorithm comparison":
        "step24_algorithm_comparison.csv",
    "25 Seed stability input":
        "step16_seed_stability.csv",
    "25 Subsampling stability input":
        "step17_subsample_stability_summary.csv",
    "25 Missingness input":
        "step20_structured_missingness_summary.csv",
    "25 Selection results":
        "step25_reliability_selection.csv",
    "26 Held-out validation":
        "step26_heldout_assignments.parquet",
    "28 Statistical validation":
        "step28_heldout_readmission_statistics.csv"
}

rows = []

for step, filename in expected.items():
    path = RESULTS / filename
    rows.append({
        "step": step,
        "file": filename,
        "exists": path.exists(),
        "size_kb": (
            round(path.stat().st_size / 1024, 2)
            if path.exists() else None
        )
    })

status = pd.DataFrame(rows)

display(status)

print("REMAINING STEP INVENTORY: PASS")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

RESULTS = PROJECT / "results"

seed = pd.read_csv(
    RESULTS / "step16_seed_stability.csv"
)

subsample = pd.read_csv(
    RESULTS / "step17_subsample_stability_summary.csv"
)

missing = pd.read_csv(
    RESULTS / "step20_structured_missingness_summary.csv"
)

algorithms = pd.read_csv(
    RESULTS / "step24_algorithm_comparison.csv"
)

heldout = pd.read_parquet(
    RESULTS / "step26_heldout_assignments.parquet"
)

statistics = pd.read_csv(
    RESULTS / "step28_heldout_readmission_statistics.csv"
)

for name, table in {
    "Seed stability": seed,
    "Subsample stability": subsample,
    "Missingness": missing,
    "Algorithm comparison": algorithms,
    "Held-out": heldout,
    "Statistics": statistics
}.items():

    print(f"\n{name}")
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())

print("\nSTEP 25A-1: PASS")

In [ ]:
print("SEED STABILITY")
display(seed)

print("\nSUBSAMPLE STABILITY")
display(subsample)

print("\n20% MISSINGNESS")
display(
    missing[
        missing["missing_rate"] == 0.20
    ]
)

print("\nALGORITHM COMPARISON")
display(
    algorithms.groupby(
        ["representation", "algorithm", "k"]
    ).agg(
        mean_silhouette=("silhouette", "mean"),
        minimum_cluster=("min_cluster", "min"),
        runs=("seed", "count")
    ).reset_index().round(4)
)

print("\nHELD-OUT CLUSTER SIZES")
for col in ["cluster_R1", "cluster_R2"]:
    print(col)
    print(heldout[col].value_counts())

print("\nHELD-OUT READMISSION STATISTICS")
display(statistics)

print("\nSTEP 25A-2: PASS")

In [ ]:
import pandas as pd
from pathlib import Path

RESULTS = Path(
    r"C:\Projects\healthcare-cluster-reliability\results"
)

seed = pd.read_csv(RESULTS / "step16_seed_stability.csv")
subsample = pd.read_csv(
    RESULTS / "step17_subsample_stability_summary.csv"
)
missing = pd.read_csv(
    RESULTS / "step20_structured_missingness_summary.csv"
)

# Normalize the candidate identifiers
seed_summary = seed[[
    "solution", "mean_ARI"
]].rename(columns={
    "mean_ARI": "seed_ARI"
})

subsample_summary = subsample[[
    "solution", "mean_ARI"
]].rename(columns={
    "mean_ARI": "subsample_ARI"
})

missing_20 = missing[
    (missing["missing_rate"] == 0.20)
    & (missing["mechanism"] == "MCAR")
].copy()

missing_20["solution"] = (
    missing_20["representation"] + "_K2"
)

missing_summary = missing_20[[
    "solution", "mean_ARI"
]].rename(columns={
    "mean_ARI": "missingness_ARI"
})

evidence = (
    seed_summary
    .merge(subsample_summary, on="solution", how="outer")
    .merge(missing_summary, on="solution", how="outer")
)

display(evidence.round(4))

evidence.to_csv(
    RESULTS / "step25_reliability_evidence_matrix.csv",
    index=False
)

print("STEP 25B: PASS")

In [ ]:
import pandas as pd
from pathlib import Path

RESULTS = Path(
    r"C:\Projects\healthcare-cluster-reliability\results"
)

evidence = pd.read_csv(
    RESULTS / "step25_reliability_evidence_matrix.csv"
)

# Prior experimental finding:
# R2_K2 matches utilization-only clustering exactly.
# R3_K2 matches R2_K2 exactly.
#
# These are development-cohort observations,
# not held-out validation results.

dominance = {
    "R1_K2": "Clinical-only representation",
    "R1_K5": "Clinical-only representation",
    "R1_K8": "Clinical-only representation",
    "R2_K2": "Utilization-only equivalent",
    "R3_K2": "Utilization-only equivalent"
}

evidence["representation_finding"] = (
    evidence["solution"].map(dominance)
)

evidence["seed_tested"] = evidence["seed_ARI"].notna()
evidence["subsample_tested"] = evidence["subsample_ARI"].notna()
evidence["missingness_tested"] = evidence["missingness_ARI"].notna()

evidence["complete_current_evidence"] = (
    evidence[
        ["seed_tested",
         "subsample_tested",
         "missingness_tested"]
    ].all(axis=1)
)

display(evidence)

evidence.to_csv(
    RESULTS / "step25c_evidence_completeness.csv",
    index=False
)

print("\nComplete candidates:")
print(
    evidence.loc[
        evidence["complete_current_evidence"],
        "solution"
    ].tolist()
)

print("\nSTEP 25C: PASS")

In [ ]:
import pandas as pd
from pathlib import Path

RESULTS = Path(
    r"C:\Projects\healthcare-cluster-reliability\results"
)

evidence = pd.read_csv(
    RESULTS / "step25c_evidence_completeness.csv"
)

# Exploratory thresholds only
MIN_SEED_ARI = 0.80
MIN_SUBSAMPLE_ARI = 0.80
MIN_MISSINGNESS_ARI = 0.70

def classify_candidate(row):

    if not row["complete_current_evidence"]:
        return "INSUFFICIENT_EVIDENCE"

    if row["representation_finding"] == "Utilization-only equivalent":
        return "FLAGGED_REPRESENTATION_DOMINANCE"

    if row["seed_ARI"] < MIN_SEED_ARI:
        return "FLAGGED_SEED_INSTABILITY"

    if row["subsample_ARI"] < MIN_SUBSAMPLE_ARI:
        return "FLAGGED_SUBSAMPLE_INSTABILITY"

    if row["missingness_ARI"] < MIN_MISSINGNESS_ARI:
        return "FLAGGED_MISSINGNESS_SENSITIVITY"

    return "ELIGIBLE"


evidence["decision"] = evidence.apply(
    classify_candidate,
    axis=1
)

display(
    evidence[[
        "solution",
        "seed_ARI",
        "subsample_ARI",
        "missingness_ARI",
        "representation_finding",
        "decision"
    ]]
)

evidence.to_csv(
    RESULTS / "step25_reliability_selection.csv",
    index=False
)

print("\nSTEP 25D: PASS")

In [ ]:
import pandas as pd
from pathlib import Path

RESULTS = Path(
    r"C:\Projects\healthcare-cluster-reliability\results"
)

evidence = pd.read_csv(
    RESULTS / "step25c_evidence_completeness.csv"
)

# Exploratory thresholds only
MIN_SEED_ARI = 0.80
MIN_SUBSAMPLE_ARI = 0.80
MIN_MISSINGNESS_ARI = 0.70

def classify_candidate(row):

    if not row["complete_current_evidence"]:
        return "INSUFFICIENT_EVIDENCE"

    if row["representation_finding"] == "Utilization-only equivalent":
        return "FLAGGED_REPRESENTATION_DOMINANCE"

    if row["seed_ARI"] < MIN_SEED_ARI:
        return "FLAGGED_SEED_INSTABILITY"

    if row["subsample_ARI"] < MIN_SUBSAMPLE_ARI:
        return "FLAGGED_SUBSAMPLE_INSTABILITY"

    if row["missingness_ARI"] < MIN_MISSINGNESS_ARI:
        return "FLAGGED_MISSINGNESS_SENSITIVITY"

    return "ELIGIBLE"


evidence["decision"] = evidence.apply(
    classify_candidate,
    axis=1
)

display(
    evidence[[
        "solution",
        "seed_ARI",
        "subsample_ARI",
        "missingness_ARI",
        "representation_finding",
        "decision"
    ]]
)

evidence.to_csv(
    RESULTS / "step25_reliability_selection.csv",
    index=False
)

print("\nSTEP 25D: PASS")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import adjusted_rand_score

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")
DATA = PROJECT / "data" / "processed"
RESULTS = PROJECT / "results"

R1 = pd.read_parquet(DATA / "development_R1_10k.parquet")
R3 = pd.read_parquet(DATA / "development_R3_10k.parquet")

assert R1["patient_nbr"].is_unique
assert R3["patient_nbr"].is_unique
assert set(R1.patient_nbr) == set(R3.patient_nbr)

R1 = R1.sort_values("patient_nbr").reset_index(drop=True)
R3 = R3.sort_values("patient_nbr").reset_index(drop=True)

diagnosis_cols = [
    "diag1_family", "diag2_family", "diag3_family"
]

candidates = {
    "R1_K5": (R1, 5),
    "R1_K8": (R1, 8),
    "R3_K2": (R3, 2)
}

def make_preprocessor(df):
    features = df.drop(columns=["patient_nbr"])

    categorical = features.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    numeric = [
        c for c in features.columns
        if c not in categorical
    ]

    return ColumnTransformer([
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="constant",
                    fill_value="missing"
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore"
                ))
            ]),
            categorical
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="median"
                )),
                ("scaler", RobustScaler())
            ]),
            numeric
        )
    ])

print("STEP 25E-1: PASS")

In [ ]:
rates = [0.05, 0.10, 0.20, 0.30]
repetitions = 10
rows = []

for solution, (df, k) in candidates.items():

    X = df.drop(columns=["patient_nbr"])
    prep = make_preprocessor(df)

    # Fit preprocessing once on original data
    X_ref = prep.fit_transform(X)

    reference = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    ).fit_predict(X_ref)

    for rate in rates:
        for rep in range(repetitions):

            rng = np.random.default_rng(
                25000 + rep + int(rate * 1000)
            )

            altered = X.copy()

            for col in diagnosis_cols:
                observed = np.flatnonzero(
                    altered[col].notna().to_numpy()
                    & altered[col].ne("missing").to_numpy()
                )

                n_mask = round(len(observed) * rate)

                chosen = rng.choice(
                    observed,
                    size=n_mask,
                    replace=False
                )

                altered.loc[chosen, col] = "missing"

            X_masked = prep.transform(altered)

            predicted = KMeans(
                n_clusters=k,
                random_state=42,
                n_init=20
            ).fit_predict(X_masked)

            rows.append({
                "solution": solution,
                "missing_rate": rate,
                "repetition": rep,
                "ARI": adjusted_rand_score(
                    reference, predicted
                )
            })

    print(solution, "completed")

runs = pd.DataFrame(rows)

runs.to_csv(
    RESULTS / "step25e_missingness_runs.csv",
    index=False
)

summary = (
    runs.groupby(["solution", "missing_rate"])
    .agg(
        mean_ARI=("ARI", "mean"),
        std_ARI=("ARI", "std"),
        min_ARI=("ARI", "min"),
        max_ARI=("ARI", "max")
    )
    .reset_index()
)

display(summary.round(4))

summary.to_csv(
    RESULTS / "step25e_missingness_summary.csv",
    index=False
)

print("STEP 25E: PASS")

In [ ]:
# STEP 25F: Harmonized missingness experiment

R2 = pd.read_parquet(
    DATA / "development_R2_10k.parquet"
)

R2 = R2.sort_values(
    "patient_nbr"
).reset_index(drop=True)

assert R1.patient_nbr.equals(R2.patient_nbr)
assert R1.patient_nbr.equals(R3.patient_nbr)

all_candidates = {
    "R1_K2": (R1, 2),
    "R1_K5": (R1, 5),
    "R1_K8": (R1, 8),
    "R2_K2": (R2, 2),
    "R3_K2": (R3, 2)
}

rates = [0.05, 0.10, 0.20, 0.30]
repetitions = 20

# Create the same patient-level masking schedule
# for all representations and candidate K values.

mask_schedule = {}

for rate in rates:
    for rep in range(repetitions):

        rng = np.random.default_rng(
            50000 + rep + int(rate * 1000)
        )

        mask_schedule[(rate, rep)] = {}

        for col in diagnosis_cols:

            observed = np.flatnonzero(
                R1[col].notna().to_numpy()
                & R1[col].ne("missing").to_numpy()
            )

            n_mask = round(
                len(observed) * rate
            )

            mask_schedule[(rate, rep)][col] = (
                rng.choice(
                    observed,
                    size=n_mask,
                    replace=False
                )
            )

rows = []

for solution, (df, k) in all_candidates.items():

    X = df.drop(columns=["patient_nbr"])

    prep = make_preprocessor(df)
    X_reference = prep.fit_transform(X)

    reference = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    ).fit_predict(X_reference)

    for rate in rates:
        for rep in range(repetitions):

            altered = X.copy()

            for col in diagnosis_cols:
                positions = mask_schedule[
                    (rate, rep)
                ][col]

                altered.loc[
                    positions, col
                ] = "missing"

            X_masked = prep.transform(altered)

            labels = KMeans(
                n_clusters=k,
                random_state=42,
                n_init=20
            ).fit_predict(X_masked)

            rows.append({
                "solution": solution,
                "missing_rate": rate,
                "repetition": rep,
                "ARI": adjusted_rand_score(
                    reference, labels
                )
            })

    print(solution, "completed")

harmonized_runs = pd.DataFrame(rows)

harmonized_summary = (
    harmonized_runs
    .groupby(["solution", "missing_rate"])
    .agg(
        mean_ARI=("ARI", "mean"),
        std_ARI=("ARI", "std"),
        min_ARI=("ARI", "min"),
        max_ARI=("ARI", "max")
    )
    .reset_index()
)

harmonized_runs.to_csv(
    RESULTS / "step25f_harmonized_missingness_runs.csv",
    index=False
)

harmonized_summary.to_csv(
    RESULTS / "step25f_harmonized_missingness_summary.csv",
    index=False
)

display(
    harmonized_summary[
        harmonized_summary.missing_rate == 0.20
    ].round(4)
)

print("STEP 25F: PASS")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import pandas as pd
import numpy as np

clinical_cols = [
    "age",
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

util_cols = [
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

observation_cols = [
    "A1Cresult_observed",
    "max_glu_serum_observed"
]

rows = []

for solution, (df, k) in all_candidates.items():

    X = df.drop(columns=["patient_nbr"])

    full_prep = make_preprocessor(df)
    X_full = full_prep.fit_transform(X)

    full_labels = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    ).fit_predict(X_full)

    blocks = {
        "clinical": clinical_cols,
        "utilization": util_cols,
        "observation": observation_cols
    }

    for block_name, columns in blocks.items():

        available = [
            c for c in columns
            if c in X.columns
        ]

        if not available:
            continue

        block_df = df[
            ["patient_nbr"] + available
        ].copy()

        block_prep = make_preprocessor(block_df)

        X_block = block_prep.fit_transform(
            block_df.drop(columns=["patient_nbr"])
        )

        block_labels = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=20
        ).fit_predict(X_block)

        rows.append({
            "solution": solution,
            "block": block_name,
            "ARI_vs_full": adjusted_rand_score(
                full_labels,
                block_labels
            )
        })

dominance = pd.DataFrame(rows)

display(
    dominance.pivot(
        index="solution",
        columns="block",
        values="ARI_vs_full"
    ).round(4)
)

dominance.to_csv(
    RESULTS / "step25g_representation_dominance.csv",
    index=False
)

print("STEP 25G-1: PASS")

In [ ]:
import pandas as pd

evidence = pd.read_csv(
    RESULTS / "step25c_evidence_completeness.csv"
)

harmonized = pd.read_csv(
    RESULTS / "step25f_harmonized_missingness_summary.csv"
)

missing20 = harmonized.loc[
    np.isclose(harmonized["missing_rate"], 0.20),
    ["solution", "mean_ARI", "std_ARI"]
].rename(columns={
    "mean_ARI": "harmonized_missingness_ARI",
    "std_ARI": "harmonized_missingness_SD"
})

updated = evidence.drop(
    columns=["missingness_ARI"],
    errors="ignore"
).merge(
    missing20,
    on="solution",
    how="left",
    validate="one_to_one"
)

updated["complete_harmonized_evidence"] = (
    updated[
        [
            "seed_ARI",
            "subsample_ARI",
            "harmonized_missingness_ARI"
        ]
    ].notna().all(axis=1)
)

display(updated[[
    "solution",
    "seed_ARI",
    "subsample_ARI",
    "harmonized_missingness_ARI",
    "harmonized_missingness_SD",
    "complete_harmonized_evidence"
]].round(4))

updated.to_csv(
    RESULTS / "step25h_reliability_evidence_matrix.csv",
    index=False
)

print("STEP 25H: PASS")

In [ ]:
from pathlib import Path
import pandas as pd

RESULTS = Path(
    r"C:\Projects\healthcare-cluster-reliability\results"
)

dominance = pd.read_csv(
    RESULTS / "step25g_representation_dominance.csv"
)

display(
    dominance.pivot(
        index="solution",
        columns="block",
        values="ARI_vs_full"
    ).round(4)
)

print("STEP 25G REVIEW: PASS")

In [ ]:
from pathlib import Path
import pandas as pd

RESULTS = Path(
    r"C:\Projects\healthcare-cluster-reliability\results"
)

evidence = pd.read_csv(
    RESULTS / "step25h_reliability_evidence_matrix.csv"
)

dominance = pd.read_csv(
    RESULTS / "step25g_representation_dominance.csv"
)

utilization = (
    dominance.loc[
        dominance["block"] == "utilization",
        ["solution", "ARI_vs_full"]
    ]
    .rename(columns={
        "ARI_vs_full": "utilization_ARI"
    })
)

selection = evidence.merge(
    utilization,
    on="solution",
    how="left",
    validate="one_to_one"
)

# Exploratory thresholds, not clinically validated.
MIN_SEED = 0.80
MIN_SUBSAMPLE = 0.80
MIN_MISSINGNESS = 0.70
MAX_UTIL_EQUIVALENCE = 0.95

def evaluate(row):
    reasons = []

    if not row["complete_harmonized_evidence"]:
        return "INSUFFICIENT_EVIDENCE", "Missing required tests"

    if (
        pd.notna(row["utilization_ARI"])
        and row["utilization_ARI"] >= MAX_UTIL_EQUIVALENCE
    ):
        reasons.append("Utilization-equivalent partition")

    if row["seed_ARI"] < MIN_SEED:
        reasons.append("Seed instability")

    if row["subsample_ARI"] < MIN_SUBSAMPLE:
        reasons.append("Subsample instability")

    if row["harmonized_missingness_ARI"] < MIN_MISSINGNESS:
        reasons.append("Missingness sensitivity")

    if reasons:
        return "FLAGGED", "; ".join(reasons)

    return "ELIGIBLE", "All exploratory checks passed"

selection[["decision", "reasons"]] = selection.apply(
    lambda row: pd.Series(evaluate(row)),
    axis=1
)

columns = [
    "solution",
    "seed_ARI",
    "subsample_ARI",
    "harmonized_missingness_ARI",
    "utilization_ARI",
    "decision",
    "reasons"
]

display(selection[columns].round(4))

selection.to_csv(
    RESULTS / "step25i_reliability_selection.csv",
    index=False
)

print("STEP 25I: PASS")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RESULTS = Path(
    r"C:\Projects\healthcare-cluster-reliability\results"
)

selection = pd.read_csv(
    RESULTS / "step25i_reliability_selection.csv"
)

quality = pd.read_csv(
    RESULTS / "step15_kmeans_internal_metrics.csv"
)

print("Quality columns:", quality.columns.tolist())
display(quality.head())

print("\nSelection columns:", selection.columns.tolist())

print("\nSTEP 25J-1: PASS")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RESULTS = Path(
    r"C:\Projects\healthcare-cluster-reliability\results"
)

quality = pd.read_csv(
    RESULTS / "step15_kmeans_internal_metrics.csv"
)

selection = pd.read_csv(
    RESULTS / "step25i_reliability_selection.csv"
)

# Match Step 15 representation names to candidate IDs
representation_map = {
    "R1_clinical": "R1",
    "R2_utilization": "R2",
    "R3_observation": "R3"
}

# Use actual representation names if they differ
print("Available representations:")
print(quality["representation"].unique())

quality["rep_short"] = (
    quality["representation"]
    .map(representation_map)
)

# Safe fallback for names beginning with R1, R2, or R3
quality["rep_short"] = quality["rep_short"].fillna(
    quality["representation"].str.extract(r"^(R[123])")[0]
)

quality["solution"] = (
    quality["rep_short"]
    + "_K"
    + quality["k"].astype(str)
)

# Restrict comparison to the five evaluated candidates
comparison = selection.merge(
    quality[
        [
            "solution",
            "silhouette",
            "calinski_harabasz",
            "davies_bouldin",
            "min_cluster_size",
            "max_cluster_size"
        ]
    ],
    on="solution",
    how="left",
    validate="one_to_one"
)

assert comparison["silhouette"].notna().all(), (
    "Some candidates have no matching Step 15 metrics."
)

display(
    comparison[
        [
            "solution",
            "silhouette",
            "seed_ARI",
            "subsample_ARI",
            "harmonized_missingness_ARI",
            "utilization_ARI",
            "decision"
        ]
    ].round(4)
)

# Conventional baselines
metrics = {
    "Silhouette-only": "silhouette",
    "Seed-stability-only": "seed_ARI",
    "Subsample-stability-only": "subsample_ARI"
}

rows = []

for method, metric in metrics.items():

    best = comparison[metric].max()

    winners = comparison.loc[
        np.isclose(comparison[metric], best, atol=1e-9),
        "solution"
    ].tolist()

    rows.append({
        "method": method,
        "selected": ", ".join(winners),
        "abstained": False,
        "selection_count": len(winners)
    })

# Reliability-aware framework
eligible = comparison.loc[
    comparison["decision"] == "ELIGIBLE",
    "solution"
].tolist()

rows.append({
    "method": "Reliability-aware",
    "selected": (
        ", ".join(eligible)
        if eligible else "NONE"
    ),
    "abstained": len(eligible) == 0,
    "selection_count": len(eligible)
})

baseline_results = pd.DataFrame(rows)

display(baseline_results)

comparison.to_csv(
    RESULTS / "step25j_candidate_comparison.csv",
    index=False
)

baseline_results.to_csv(
    RESULTS / "step25j_selection_baselines.csv",
    index=False
)

print("STEP 25J-2: PASS")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

DATA = PROJECT / "data" / "processed"
RESULTS = PROJECT / "results"

development = pd.read_parquet(
    DATA / "development_R1_10k.parquet"
)

heldout = pd.read_parquet(
    RESULTS / "step26_heldout_assignments.parquet"
)

print("Held-out shape:", heldout.shape)
print("Held-out columns:", heldout.columns.tolist())

assert "patient_nbr" in heldout.columns
assert heldout["patient_nbr"].is_unique

overlap = set(development["patient_nbr"]) & set(
    heldout["patient_nbr"]
)

print("Development patients:", len(development))
print("Held-out patients:", len(heldout))
print("Overlapping patients:", len(overlap))

assert len(overlap) == 0, "Patient leakage detected"

for col in ["cluster_R1", "cluster_R2"]:
    if col in heldout.columns:
        print(f"\n{col} distribution:")
        print(
            heldout[col].value_counts(
                normalize=True
            ).round(4)
        )

print("\nSTEP 26B-1: PASS")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS = Path(
    r"C:\Projects\healthcare-cluster-reliability\results"
)

heldout = pd.read_parquet(
    RESULTS / "step26_heldout_assignments.parquet"
)

# Validate labels and outcomes
assert heldout["readmit_30d"].notna().all()
assert set(heldout["readmit_30d"].unique()).issubset({0, 1})

for col in ["cluster_R1", "cluster_R2"]:
    assert heldout[col].notna().all()
    assert set(heldout[col].unique()).issubset({0, 1})

rows = []

for representation, col in [
    ("R1", "cluster_R1"),
    ("R2", "cluster_R2")
]:
    for cluster, group in heldout.groupby(col):
        rows.append({
            "representation": representation,
            "cluster": cluster,
            "patients": len(group),
            "readmissions": int(group["readmit_30d"].sum()),
            "readmission_rate": group["readmit_30d"].mean()
        })

summary = pd.DataFrame(rows)

display(summary.round(5))

summary.to_csv(
    RESULTS / "step26b_heldout_cluster_outcomes.csv",
    index=False
)

print("\nSTEP 26B-2: PASS")

In [ ]:
from pathlib import Path

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

NOTEBOOKS = PROJECT / "notebooks"

for filename in [
    "04_clustering_baselines.ipynb",
    "06_clinical_phenotype_validation.ipynb",
    "10_heldout_validation.ipynb",
    "13_complete_reliability_validation.ipynb"
]:
    path = NOTEBOOKS / filename

    print(f"\n{'=' * 65}")
    print(filename, "| exists:", path.exists())

    if not path.exists():
        continue

    import json

    notebook = json.loads(
        path.read_text(encoding="utf-8")
    )

    for i, cell in enumerate(notebook["cells"]):
        source = "".join(cell.get("source", []))

        terms = [
            "SimpleImputer(",
            "OneHotEncoder(",
            "RobustScaler(",
            "KMeans(",
            "fit_predict(",
            ".predict("
        ]

        matches = [
            line.strip()
            for line in source.splitlines()
            if any(term in line for term in terms)
        ]

        if matches:
            print(f"\nCell {i}:")
            for line in matches[:12]:
                print(" ", line)

print("\nSTEP 26B-3 AUDIT: COMPLETE")

In [ ]:
from pathlib import Path

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

import os

print("Current working directory:")
print(os.getcwd())

print("\nProject exists:", PROJECT.exists())

print("\nSearching for notebooks...")

notebooks = sorted(
    PROJECT.rglob("*.ipynb")
) if PROJECT.exists() else []

for notebook in notebooks:
    print(notebook.relative_to(PROJECT))

print("\nTotal notebooks found:", len(notebooks))

print("\nSTEP 26B-3A: COMPLETE")

In [ ]:
import json

terms = [
    "SimpleImputer(",
    "OneHotEncoder(",
    "RobustScaler(",
    "KMeans(",
    "fit_predict(",
    ".predict("
]

for path in notebooks:

    print("\n" + "=" * 65)
    print(path.relative_to(PROJECT))

    try:
        nb = json.loads(
            path.read_text(encoding="utf-8")
        )

        for i, cell in enumerate(nb.get("cells", [])):

            source = "".join(
                cell.get("source", [])
            )

            matches = [
                line.strip()
                for line in source.splitlines()
                if any(term in line for term in terms)
            ]

            if matches:
                print(f"\nCell {i}")

                for line in matches[:15]:
                    print(" ", line)

    except Exception as e:
        print("Could not read:", e)

print("\nSTEP 26B-3B: COMPLETE")

In [ ]:
import json

for path in notebooks:
    nb = json.loads(path.read_text(encoding="utf-8"))

    for cell_number in [58]:
        if cell_number >= len(nb["cells"]):
            continue

        source = "".join(
            nb["cells"][cell_number].get("source", [])
        )

        if "heldout_assignments" in source:
            print("\nNOTEBOOK:", path.name)
            print("\nCELL:", cell_number)
            print(source)

print("\nSTEP 26B-3C: INSPECTION COMPLETE")

In [ ]:
import json

for path in notebooks:
    nb = json.loads(path.read_text(encoding="utf-8"))

    for cell_number in [58]:
        if cell_number >= len(nb["cells"]):
            continue

        source = "".join(
            nb["cells"][cell_number].get("source", [])
        )

        if "heldout_assignments" in source:
            print("\nNOTEBOOK:", path.name)
            print("\nCELL:", cell_number)
            print(source)

print("\nSTEP 26B-3C: INSPECTION COMPLETE")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

def make_heldout_preprocessor(rep):

    CLIN = [
        "age",
        "diag1_family",
        "diag2_family",
        "diag3_family"
    ]

    UTIL = [
        "number_outpatient",
        "number_emergency",
        "number_inpatient"
    ]

    blocks = [
        (
            "clinical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="missing"
                    )
                ),
                (
                    "encoder",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            CLIN
        )
    ]

    if rep == "R2":
        blocks.append(
            (
                "util",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(strategy="median")
                    ),
                    ("scaler", RobustScaler())
                ]),
                UTIL
            )
        )

    return ColumnTransformer(blocks)


print("PREPROCESSOR FIX: PASS")

In [ ]:

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# --------------------------------------------------
# Configuration
# --------------------------------------------------
rates = [0.05, 0.10, 0.20, 0.30]
repetitions = 20

diagnosis_cols = [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

# Verify the correct held-out dataset
assert len(heldout_features) == 61518
assert all(
    col in heldout_features.columns
    for col in diagnosis_cols
)

rows = []

# --------------------------------------------------
# Run experiments for R1 and R2
# --------------------------------------------------
for rep in ["R1", "R2"]:

    print(f"\nRunning {rep}...")

    # Load development data
    train = pd.read_parquet(
        DATA / f"development_{rep}_10k.parquet"
    )

    # Use the uniquely named preprocessing function
    prep = make_heldout_preprocessor(rep)

    # Fit preprocessing ONLY on development data
    X_train = prep.fit_transform(train)

    # Transform held-out data without refitting
    X_test = prep.transform(heldout_features)

    # Fit clustering model ONLY on development data
    model = KMeans(
        n_clusters=2,
        random_state=42,
        n_init=20
    )

    model.fit(X_train)

    # Original held-out assignments
    reference = model.predict(X_test)

    # --------------------------------------------------
    # Simulate diagnosis missingness
    # --------------------------------------------------
    for rate in rates:

        for iteration in range(repetitions):

            rng = np.random.default_rng(
                26000 + iteration + int(rate * 1000)
            )

            # IMPORTANT: Use full held-out features
            altered = heldout_features.copy()

            for col in diagnosis_cols:

                # Select currently observed diagnosis entries
                observed = np.flatnonzero(
                    altered[col].notna().to_numpy()
                    & altered[col].ne("missing").to_numpy()
                )

                n_mask = round(len(observed) * rate)

                chosen = rng.choice(
                    observed,
                    size=n_mask,
                    replace=False
                )

                # Mask selected diagnosis entries
                altered.iloc[
                    chosen,
                    altered.columns.get_loc(col)
                ] = "missing"

            # Transform masked data using fitted preprocessor
            X_masked = prep.transform(altered)

            # Predict using the SAME fitted model
            predicted = model.predict(X_masked)

            # Record robustness metrics
            rows.append({
                "representation": rep,
                "missing_rate": rate,
                "iteration": iteration,
                "ARI": adjusted_rand_score(
                    reference,
                    predicted
                ),
                "fraction_changed": float(
                    np.mean(reference != predicted)
                )
            })

        print(f"  {int(rate * 100)}% missingness completed")

    print(f"{rep} completed")

# --------------------------------------------------
# Create summary
# --------------------------------------------------
heldout_robustness = pd.DataFrame(rows)

summary = (
    heldout_robustness
    .groupby(["representation", "missing_rate"])
    .agg(
        mean_ARI=("ARI", "mean"),
        std_ARI=("ARI", "std"),
        mean_fraction_changed=(
            "fraction_changed", "mean"
        )
    )
    .reset_index()
)

print("\nHELD-OUT MISSINGNESS ROBUSTNESS SUMMARY")
display(summary.round(4))

# --------------------------------------------------
# Save results
# --------------------------------------------------
heldout_robustness.to_csv(
    RESULTS / "step26b_heldout_missingness_runs.csv",
    index=False
)

summary.to_csv(
    RESULTS / "step26b_heldout_missingness_summary.csv",
    index=False
)

assert len(heldout_robustness) == 160
assert len(summary) == 8

print("\nSTEP 26B-4: PASS")


In [ ]:
from sklearn.metrics import adjusted_rand_score
from sklearn.cluster import KMeans

saved = pd.read_parquet(
    RESULTS / "step26_heldout_assignments.parquet"
)

for rep in ["R1", "R2"]:
    train = pd.read_parquet(
        DATA / f"development_{rep}_10k.parquet"
    )

    prep = make_heldout_preprocessor(rep)
    X_train = prep.fit_transform(train)
    X_test = prep.transform(heldout_features)

    model = KMeans(n_clusters=2, random_state=42, n_init=20)
    model.fit(X_train)

    predicted = pd.DataFrame({
        "patient_nbr": heldout_features["patient_nbr"],
        "predicted": model.predict(X_test)
    })

    comparison = predicted.merge(
        saved[["patient_nbr", f"cluster_{rep}"]],
        on="patient_nbr",
        validate="one_to_one"
    )

    assert len(comparison) == len(heldout_features)

    ari = adjusted_rand_score(
        comparison["predicted"],
        comparison[f"cluster_{rep}"]
    )

    print(f"{rep} saved-assignment reproduction ARI: {ari:.6f}")
    assert abs(ari - 1.0) < 1e-10

print("STEP 26B-4 MODEL REPRODUCTION: PASS")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")
DATA = PROJECT / "data" / "processed"
RESULTS = PROJECT / "results"

# Full patient-level dataset
full_data = pd.read_parquet(
    DATA / "uci_diabetes_index_patient_features_v2.parquet"
)

# Development patient IDs
development = pd.read_parquet(
    DATA / "development_R1_10k.parquet"
)

development_ids = set(development["patient_nbr"])

# Restore held-out patients WITH all features
heldout_features = full_data.loc[
    ~full_data["patient_nbr"].isin(development_ids)
].copy().reset_index(drop=True)

required = [
    "patient_nbr",
    "age",
    "diag1_family",
    "diag2_family",
    "diag3_family",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "readmit_30d"
]

assert all(c in heldout_features.columns for c in required)
assert len(heldout_features) == 61518
assert heldout_features["patient_nbr"].is_unique

print("Held-out feature shape:", heldout_features.shape)
print("Held-out patients:", len(heldout_features))
print("All required features available: PASS")

In [ ]:
for rep in ["R1", "R2"]:

    train = pd.read_parquet(
        DATA / f"development_{rep}_10k.parquet"
    )

    prep = make_heldout_preprocessor(rep)

    X_train = prep.fit_transform(train)
    X_test = prep.transform(heldout_features)

    print(
        rep,
        "Train:", X_train.shape,
        "Held-out:", X_test.shape
    )

    assert X_train.shape[1] == X_test.shape[1]

print("\nSTEP 26B-4 PREPROCESSING: PASS")

In [ ]:

import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import adjusted_rand_score

UTIL = [
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

# Full R2 model fitted on development patients
train = pd.read_parquet(
    DATA / "development_R2_10k.parquet"
)

prep_full = make_heldout_preprocessor("R2")
X_train_full = prep_full.fit_transform(train)
X_test_full = prep_full.transform(heldout_features)

model_full = KMeans(
    n_clusters=2, random_state=42, n_init=20
)
model_full.fit(X_train_full)

labels_full = model_full.predict(X_test_full)

# Utilization-only model fitted on development patients
prep_util = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

X_train_util = prep_util.fit_transform(train[UTIL])
X_test_util = prep_util.transform(heldout_features[UTIL])

model_util = KMeans(
    n_clusters=2, random_state=42, n_init=20
)
model_util.fit(X_train_util)

labels_util = model_util.predict(X_test_util)

# Compare held-out partitions
ari = adjusted_rand_score(labels_full, labels_util)

print("Held-out R2 vs utilization-only ARI:", round(ari, 6))

comparison = pd.crosstab(
    pd.Series(labels_full, name="R2_full"),
    pd.Series(labels_util, name="Utilization_only")
)

display(comparison)

result = pd.DataFrame([{
    "representation": "R2_K2",
    "comparison": "Full vs utilization-only",
    "heldout_ARI": ari,
    "patients": len(heldout_features)
}])

result.to_csv(
    RESULTS / "step26b_heldout_representation_dominance.csv",
    index=False
)

print("STEP 26B-5: COMPLETE")


In [ ]:

from pathlib import Path

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")
MIMIC = PROJECT / "data" / "raw" / "mimic"

print("MIMIC folder exists:", MIMIC.exists())

if MIMIC.exists():
    files = sorted(
        p for p in MIMIC.rglob("*")
        if p.is_file()
    )

    print("Total files:", len(files))

    for path in files[:40]:
        print(
            path.relative_to(MIMIC),
            "|",
            round(path.stat().st_size / (1024**2), 2),
            "MB"
        )
else:
    print("No local MIMIC-IV data found.")

print("\nSTEP 27A: INVENTORY COMPLETE")


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")

MIMIC = (
    PROJECT
    / "data"
    / "raw"
    / "mimic"
    / "mimic-iv-clinical-database-demo-2.2"
    / "mimic-iv-clinical-database-demo-2.2"
    / "hosp"
)

# Load core tables
patients = pd.read_csv(
    MIMIC / "patients.csv.gz"
)

admissions = pd.read_csv(
    MIMIC / "admissions.csv.gz"
)

diagnoses = pd.read_csv(
    MIMIC / "diagnoses_icd.csv.gz"
)

diagnosis_dictionary = pd.read_csv(
    MIMIC / "d_icd_diagnoses.csv.gz"
)

# Basic population counts
print("MIMIC-IV DEMO POPULATION")
print("-" * 50)

print("Patients:", patients["subject_id"].nunique())
print("Admissions:", len(admissions))
print("Patients with admissions:",
      admissions["subject_id"].nunique())

print("Diagnosis rows:", len(diagnoses))
print("Admissions with diagnoses:",
      diagnoses["hadm_id"].nunique())

# Diabetes ICD-9 and ICD-10 codes
codes = diagnoses["icd_code"].astype(str).str.upper()

diabetes_mask = (
    (
        diagnoses["icd_version"].eq(9)
        & codes.str.startswith("250")
    )
    |
    (
        diagnoses["icd_version"].eq(10)
        & codes.str.match(r"^E(08|09|10|11|13)")
    )
)

diabetes_diagnoses = diagnoses.loc[diabetes_mask]

print("\nDIABETES COHORT")
print("-" * 50)

print(
    "Patients with diabetes diagnosis:",
    diabetes_diagnoses["subject_id"].nunique()
)

print(
    "Admissions with diabetes diagnosis:",
    diabetes_diagnoses["hadm_id"].nunique()
)

# Check chronological information
print("\nADMISSION DATE FIELDS")
print("-" * 50)

for col in ["admittime", "dischtime", "deathtime"]:
    print(col, "available:", col in admissions.columns)

# Inspect schemas
print("\nPATIENT COLUMNS")
print(patients.columns.tolist())

print("\nADMISSION COLUMNS")
print(admissions.columns.tolist())

print("\nDIAGNOSIS COLUMNS")
print(diagnoses.columns.tolist())

print("\nSTEP 27B: COMPLETE")


In [ ]:

import pandas as pd
import numpy as np

# --------------------------------------------------
# Step 27C: Chronological patient-level construction
# --------------------------------------------------

admissions2 = admissions.copy()
diagnoses2 = diagnoses.copy()

admissions2["admittime"] = pd.to_datetime(
    admissions2["admittime"], errors="coerce"
)

admissions2["dischtime"] = pd.to_datetime(
    admissions2["dischtime"], errors="coerce"
)

# Identify diabetes-coded admissions
codes = diagnoses2["icd_code"].astype(str).str.upper()

diabetes_mask = (
    (
        diagnoses2["icd_version"].eq(9)
        & codes.str.startswith("250")
    )
    |
    (
        diagnoses2["icd_version"].eq(10)
        & codes.str.match(r"^E(08|09|10|11|13)")
    )
)

diabetes_hadm = set(
    diagnoses2.loc[diabetes_mask, "hadm_id"]
)

# Sort admissions chronologically
admissions2 = admissions2.sort_values(
    ["subject_id", "admittime", "hadm_id"]
).reset_index(drop=True)

# Count earlier admissions in available data
admissions2["prior_admissions"] = (
    admissions2.groupby("subject_id").cumcount()
)

# Select first diabetes-coded admission per patient
index_admissions = (
    admissions2[
        admissions2["hadm_id"].isin(diabetes_hadm)
    ]
    .drop_duplicates("subject_id", keep="first")
    .copy()
)

# Add patient demographics
mimic_patient_level = index_admissions.merge(
    patients[
        ["subject_id", "gender", "anchor_age", "anchor_year"]
    ],
    on="subject_id",
    how="left",
    validate="many_to_one"
)

# Number of diagnoses on index admission
index_diagnosis_counts = (
    diagnoses2.groupby("hadm_id")
    .size()
    .rename("index_diagnosis_count")
    .reset_index()
)

mimic_patient_level = mimic_patient_level.merge(
    index_diagnosis_counts,
    on="hadm_id",
    how="left",
    validate="one_to_one"
)

# Observation flags
mimic_patient_level["age_observed"] = (
    mimic_patient_level["anchor_age"].notna()
)

mimic_patient_level["diagnosis_observed"] = (
    mimic_patient_level["index_diagnosis_count"].notna()
)

# Check uniqueness and cohort size
assert mimic_patient_level["subject_id"].is_unique
assert len(mimic_patient_level) == 35

print("MIMIC PATIENT-LEVEL DATASET")
print("-" * 50)

print("Patients:", len(mimic_patient_level))
print(
    "Prior admission distribution:"
)
print(
    mimic_patient_level["prior_admissions"]
    .describe()
)

print("\nMissingness summary:")
print(
    mimic_patient_level[
        ["age_observed", "diagnosis_observed"]
    ].mean()
)

# Save
mimic_patient_level.to_parquet(
    DATA / "step27c_mimic_diabetes_patient_level.parquet",
    index=False
)

print("\nSTEP 27C: PASS")


In [ ]:

import pandas as pd

# Load UCI development representation
uci = pd.read_parquet(
    DATA / "development_R1_10k.parquet"
)

diagnosis_cols = [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

print("UCI DIAGNOSIS FAMILY VOCABULARY")
print("=" * 55)

for col in diagnosis_cols:
    print(f"\n{col}")
    print("-" * 40)

    counts = (
        uci[col]
        .fillna("MISSING")
        .astype(str)
        .value_counts()
    )

    print(counts.to_string())

# Inspect MIMIC diagnosis coding distribution
print("\nMIMIC DIAGNOSIS CODING")
print("=" * 55)

print(
    diagnoses["icd_version"]
    .value_counts(dropna=False)
)

# Display common MIMIC diagnoses in the diabetes cohort
index_ids = set(mimic_patient_level["hadm_id"])

index_diagnoses = diagnoses[
    diagnoses["hadm_id"].isin(index_ids)
].copy()

common_codes = (
    index_diagnoses
    .groupby(["icd_version", "icd_code"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

print("\nTOP 20 INDEX-ADMISSION DIAGNOSIS CODES")
print(common_codes.to_string())

print("\nSTEP 27D-1: COMPLETE")


In [ ]:

import pandas as pd
import numpy as np

# --------------------------------------------------
# Step 27D-2: Conservative ICD-9 family mapping
# --------------------------------------------------

def map_icd9_family(code):
    if pd.isna(code):
        return "missing"

    code = str(code).strip().upper().replace(".", "")

    if not code:
        return "missing"

    if code.startswith("V"):
        return "supplementary_V"

    if code.startswith("E"):
        return "external_E"

    if not code[:3].isdigit():
        return "unmapped_review"

    chapter = int(code[:3])

    ranges = [
        (1, 139, "infectious"),
        (140, 239, "neoplasm"),
        (240, 279, "endocrine_metabolic"),
        (280, 289, "blood"),
        (290, 319, "mental"),
        (320, 389, "nervous_sensory"),
        (390, 459, "circulatory"),
        (460, 519, "respiratory"),
        (520, 579, "digestive"),
        (580, 629, "genitourinary"),
        (630, 679, "pregnancy"),
        (680, 709, "skin"),
        (710, 739, "musculoskeletal"),
        (740, 759, "congenital"),
        (760, 779, "perinatal"),
        (780, 799, "symptoms_signs"),
        (800, 999, "injury_poisoning")
    ]

    for low, high, family in ranges:
        if low <= chapter <= high:
            return family

    return "unmapped_review"


def map_mimic_diagnosis(code, version):

    if pd.isna(code):
        return "missing", "missing_code"

    if version == 9:
        family = map_icd9_family(code)
        status = (
            "mapped_icd9"
            if family not in ["unmapped_review", "missing"]
            else "unmapped_review"
        )
        return family, status

    if version == 10:
        return "unmapped_review", "icd10_pending_mapping"

    return "unmapped_review", "unknown_icd_version"


# Use only diagnoses from selected index admissions
mapped = index_diagnoses.copy()

mapped[["diagnosis_family", "mapping_status"]] = (
    mapped.apply(
        lambda row: pd.Series(
            map_mimic_diagnosis(
                row["icd_code"],
                row["icd_version"]
            )
        ),
        axis=1
    )
)

# --------------------------------------------------
# Coverage summary
# --------------------------------------------------

coverage = (
    mapped["mapping_status"]
    .value_counts()
    .rename_axis("mapping_status")
    .reset_index(name="diagnosis_rows")
)

coverage["percentage"] = (
    coverage["diagnosis_rows"]
    / len(mapped)
    * 100
)

print("MIMIC DIAGNOSIS MAPPING COVERAGE")
display(coverage.round(2))

print("\nMAPPED DIAGNOSIS FAMILIES")
display(
    mapped["diagnosis_family"]
    .value_counts()
    .rename_axis("family")
    .reset_index(name="rows")
)

# --------------------------------------------------
# Save auditable mapping
# --------------------------------------------------

mapped.to_csv(
    RESULTS / "step27d_mimic_diagnosis_mapping.csv",
    index=False
)

coverage.to_csv(
    RESULTS / "step27d_mimic_mapping_coverage.csv",
    index=False
)

print("\nSTEP 27D-2: COMPLETE")


In [ ]:

import pandas as pd

# Select unmapped diagnoses
unmapped = mapped[
    mapped["diagnosis_family"] == "unmapped_review"
].copy()

# Join diagnosis descriptions
unmapped = unmapped.merge(
    diagnosis_dictionary[
        ["icd_code", "icd_version", "long_title"]
    ],
    on=["icd_code", "icd_version"],
    how="left",
    validate="many_to_one"
)

# Aggregate distinct codes
unmapped_codes = (
    unmapped
    .groupby(
        ["icd_version", "icd_code", "long_title"],
        dropna=False
    )
    .size()
    .reset_index(name="diagnosis_rows")
    .sort_values(
        ["diagnosis_rows", "icd_code"],
        ascending=[False, True]
    )
)

print("UNMAPPED DIAGNOSIS SUMMARY")
print("-" * 50)

print("Unmapped diagnosis rows:", len(unmapped))
print("Distinct unmapped codes:", len(unmapped_codes))

print("\nUnmapped by ICD version:")
print(
    unmapped["icd_version"].value_counts()
)

print("\nTOP 30 UNMAPPED DIAGNOSES")
display(unmapped_codes.head(30))

# Save review inventory
unmapped_codes.to_csv(
    RESULTS / "step27d_unmapped_code_review.csv",
    index=False
)

print("\nSTEP 27D-3: COMPLETE")


In [ ]:

import pandas as pd
import numpy as np

# --------------------------------------------------
# Step 27D-4: ICD-10 chapter-level mapping
# --------------------------------------------------

def map_icd10_family(code):

    if pd.isna(code):
        return "missing"

    code = str(code).strip().upper().replace(".", "")

    if not code:
        return "missing"

    letter = code[0]

    if letter in ["A", "B"]:
        return "infectious"

    if letter == "C":
        return "neoplasm"

    if letter == "D":
        if len(code) < 3 or not code[1:3].isdigit():
            return "unmapped_review"

        number = int(code[1:3])

        if number <= 49:
            return "neoplasm"

        if 50 <= number <= 89:
            return "blood"

        return "unmapped_review"

    family_map = {
        "E": "endocrine_metabolic",
        "F": "mental",
        "G": "nervous_sensory",
        "H": "nervous_sensory",
        "I": "circulatory",
        "J": "respiratory",
        "K": "digestive",
        "L": "skin",
        "M": "musculoskeletal",
        "N": "genitourinary",
        "O": "pregnancy",
        "P": "perinatal",
        "Q": "congenital",
        "R": "symptoms_signs",
        "S": "injury_poisoning",
        "T": "injury_poisoning",
        "V": "external_E",
        "W": "external_E",
        "X": "external_E",
        "Y": "external_E",
        "Z": "supplementary_V"
    }

    return family_map.get(
        letter,
        "unmapped_review"
    )


def map_diagnosis_family_v2(code, version):

    if pd.isna(code):
        return "missing", "missing_code"

    if version == 9:
        family = map_icd9_family(code)

        status = (
            "mapped_icd9"
            if family not in ["missing", "unmapped_review"]
            else "unmapped_review"
        )

        return family, status

    if version == 10:
        family = map_icd10_family(code)

        status = (
            "mapped_icd10_provisional"
            if family not in ["missing", "unmapped_review"]
            else "unmapped_review"
        )

        return family, status

    return "unmapped_review", "unknown_icd_version"


# --------------------------------------------------
# Apply mapping
# --------------------------------------------------

mapped_v2 = index_diagnoses.copy()

mapped_v2[
    ["diagnosis_family", "mapping_status"]
] = mapped_v2.apply(
    lambda row: pd.Series(
        map_diagnosis_family_v2(
            row["icd_code"],
            row["icd_version"]
        )
    ),
    axis=1
)

# --------------------------------------------------
# Coverage and validation
# --------------------------------------------------

coverage_v2 = (
    mapped_v2["mapping_status"]
    .value_counts()
    .rename_axis("mapping_status")
    .reset_index(name="diagnosis_rows")
)

coverage_v2["percentage"] = (
    coverage_v2["diagnosis_rows"]
    / len(mapped_v2)
    * 100
)

print("ICD-9 / ICD-10 MAPPING COVERAGE")
display(coverage_v2.round(2))

print("\nDIAGNOSIS FAMILY DISTRIBUTION")
display(
    mapped_v2["diagnosis_family"]
    .value_counts()
    .rename_axis("family")
    .reset_index(name="rows")
)

# Check any remaining unmapped records
remaining = mapped_v2[
    mapped_v2["diagnosis_family"] == "unmapped_review"
]

print("\nRemaining unmapped records:", len(remaining))

# Save
mapped_v2.to_csv(
    RESULTS / "step27d_mimic_diagnosis_mapping_v2.csv",
    index=False
)

coverage_v2.to_csv(
    RESULTS / "step27d_mimic_mapping_coverage_v2.csv",
    index=False
)

print("\nSTEP 27D-4: COMPLETE")


In [ ]:

import pandas as pd
import numpy as np

# --------------------------------------------------
# Step 27E: Patient-level diagnosis harmonization
# --------------------------------------------------

# Keep only diagnoses from the 35 index admissions
diagnosis_index = mapped_v2[
    mapped_v2["hadm_id"].isin(
        mimic_patient_level["hadm_id"]
    )
].copy()

# Check diagnosis sequence uniqueness
assert not diagnosis_index.duplicated(
    ["hadm_id", "seq_num"]
).any(), "Duplicate diagnosis sequence numbers found"

diagnosis_index = diagnosis_index.sort_values(
    ["hadm_id", "seq_num"]
)

# Keep first three recorded diagnosis positions
first_three = diagnosis_index[
    diagnosis_index["seq_num"].between(1, 3)
].copy()

# Create patient-level wide diagnosis features
wide = first_three.pivot(
    index="hadm_id",
    columns="seq_num",
    values=[
        "diagnosis_family",
        "icd_code",
        "icd_version",
        "mapping_status"
    ]
)

wide.columns = [
    f"{field}_{int(position)}"
    for field, position in wide.columns
]

wide = wide.reset_index()

# Rename diagnosis-family fields
wide = wide.rename(columns={
    "diagnosis_family_1": "diag1_family",
    "diagnosis_family_2": "diag2_family",
    "diagnosis_family_3": "diag3_family"
})

# Merge into patient-level dataset
mimic_harmonized = mimic_patient_level.merge(
    wide,
    on="hadm_id",
    how="left",
    validate="one_to_one"
)

# Preserve missingness explicitly
for col in [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]:
    if col not in mimic_harmonized.columns:
        mimic_harmonized[col] = pd.NA

    mimic_harmonized[col] = (
        mimic_harmonized[col]
        .fillna("missing")
    )

# Add provenance
mimic_harmonized["source_dataset"] = (
    "MIMIC-IV demo 2.2"
)

mimic_harmonized["index_definition"] = (
    "first diabetes-coded admission in available records"
)

mimic_harmonized["harmonization_version"] = (
    "provisional_icd_chapter_v2"
)

# Validation
assert len(mimic_harmonized) == 35
assert mimic_harmonized["subject_id"].is_unique

print("HARMONIZED MIMIC PATIENT DATASET")
print("-" * 50)

print("Patients:", len(mimic_harmonized))

print("\nDiagnosis-family coverage:")
for col in [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]:
    print(
        col,
        mimic_harmonized[col]
        .ne("missing")
        .sum(),
        "/",
        len(mimic_harmonized)
    )

print("\nDiagnosis-family distributions:")
for col in [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]:
    print(f"\n{col}")
    print(
        mimic_harmonized[col]
        .value_counts()
        .head(10)
    )

# Save
mimic_harmonized.to_parquet(
    DATA / "step27e_mimic_harmonized_patient_features.parquet",
    index=False
)

print("\nSTEP 27E: PASS")


In [ ]:

import pandas as pd
import numpy as np

# --------------------------------------------------
# Step 27F: Harmonization integrity audit
# --------------------------------------------------

uci = pd.read_parquet(
    DATA / "development_R1_10k.parquet"
)

diagnosis_cols = [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

# UCI vocabulary
uci_vocab = set()

for col in diagnosis_cols:
    uci_vocab.update(
        uci[col].dropna().astype(str).unique()
    )

# MIMIC vocabulary
mimic_vocab = set()

for col in diagnosis_cols:
    mimic_vocab.update(
        mimic_harmonized[col]
        .dropna()
        .astype(str)
        .unique()
    )

# Identify incompatible categories
unexpected = sorted(
    mimic_vocab - uci_vocab
)

print("DIAGNOSIS VOCABULARY AUDIT")
print("-" * 50)

print("UCI families:", len(uci_vocab))
print("MIMIC families:", len(mimic_vocab))
print("Unexpected MIMIC families:", unexpected)

# Verify original ICD code and mapping status
for position in [1, 2, 3]:

    family_col = f"diag{position}_family"
    code_col = f"icd_code_{position}"
    version_col = f"icd_version_{position}"
    status_col = f"mapping_status_{position}"

    assert all(
        col in mimic_harmonized.columns
        for col in [
            family_col,
            code_col,
            version_col,
            status_col
        ]
    )

    # A populated family must have a source code
    invalid = (
        mimic_harmonized[family_col].ne("missing")
        & mimic_harmonized[code_col].isna()
    )

    assert not invalid.any(), (
        f"Missing ICD provenance at position {position}"
    )

    print(
        f"Position {position}:",
        "provenance PASS"
    )

# Confirm patient uniqueness
assert mimic_harmonized["subject_id"].is_unique

# Summary
audit = pd.DataFrame([{
    "patients": len(mimic_harmonized),
    "uci_family_count": len(uci_vocab),
    "mimic_family_count": len(mimic_vocab),
    "unexpected_family_count": len(unexpected),
    "unexpected_families": ",".join(unexpected),
    "provenance_present": True
}])

audit.to_csv(
    RESULTS / "step27f_harmonization_audit.csv",
    index=False
)

print("\nSTEP 27F: COMPLETE")


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")
RESULTS = PROJECT / "results"

expected_files = {
    "Internal cluster quality":
        "step15_kmeans_internal_metrics.csv",
    "Seed stability":
        "step16_seed_stability.csv",
    "Subsample stability":
        "step17_subsample_stability_summary.csv",
    "Development missingness":
        "step25f_harmonized_missingness_summary.csv",
    "Development representation dominance":
        "step25g_representation_dominance.csv",
    "Reliability selection":
        "step25i_reliability_selection.csv",
    "Selection baseline comparison":
        "step25j_selection_baselines.csv",
    "Held-out missingness":
        "step26b_heldout_missingness_summary.csv",
    "Held-out representation dominance":
        "step26b_heldout_representation_dominance.csv",
    "Held-out readmission statistics":
        "step28_heldout_readmission_statistics.csv",
    "MIMIC harmonization audit":
        "step27f_harmonization_audit.csv"
}

inventory = []

for analysis, filename in expected_files.items():

    path = RESULTS / filename
    exists = path.exists()

    rows = None
    columns = None

    if exists:
        df = pd.read_csv(path)
        rows = len(df)
        columns = ", ".join(df.columns.tolist())

    inventory.append({
        "analysis": analysis,
        "filename": filename,
        "exists": exists,
        "rows": rows,
        "columns": columns
    })

inventory_df = pd.DataFrame(inventory)

display(
    inventory_df[
        ["analysis", "exists", "rows"]
    ]
)

inventory_df.to_csv(
    RESULTS / "step28b_evidence_inventory.csv",
    index=False
)

print(
    "\nFiles available:",
    inventory_df["exists"].sum(),
    "/",
    len(inventory_df)
)

print("\nSTEP 28B-1: COMPLETE")


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")
RESULTS = PROJECT / "results"

# Use the filenames confirmed in Step 28B-1
inventory = pd.read_csv(
    RESULTS / "step28b_evidence_inventory.csv"
)

# Inspect the main reliability experiments
analyses = [
    "Internal cluster quality",
    "Seed stability",
    "Subsample stability",
    "Development missingness",
    "Development representation dominance",
    "Reliability selection"
]

for analysis in analyses:

    row = inventory.loc[
        inventory["analysis"] == analysis
    ]

    if row.empty:
        print(f"\nNOT FOUND IN INVENTORY: {analysis}")
        continue

    filename = row.iloc[0]["filename"]
    df = pd.read_csv(RESULTS / filename)

    print("\n" + "=" * 65)
    print(analysis.upper())
    print("File:", filename)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    print("\nFirst five rows:")
    display(df.head(5))

print("\nSTEP 28B-2: COMPLETE")


In [ ]:

import pandas as pd
import numpy as np

# --------------------------------------------------
# Step 28B-3: Manuscript Table 2
# --------------------------------------------------

selection = pd.read_csv(
    RESULTS / "step25i_reliability_selection.csv"
)

internal = pd.read_csv(
    RESULTS / "step15_kmeans_internal_metrics.csv"
)

print("Step 15 columns:", internal.columns.tolist())

# Normalize candidate identifiers if already present
if "solution" in internal.columns:
    internal["solution"] = (
        internal["solution"].astype(str)
    )

# Show Step 15 structure before selecting silhouette
display(internal.head(10))

# Core reliability metrics from validated Step 25I
table2 = selection[
    [
        "solution",
        "seed_ARI",
        "subsample_ARI",
        "harmonized_missingness_ARI",
        "harmonized_missingness_SD",
        "utilization_ARI",
        "decision",
        "reasons"
    ]
].copy()

table2 = table2.rename(columns={
    "solution": "Candidate",
    "seed_ARI": "Seed ARI",
    "subsample_ARI": "Subsample ARI",
    "harmonized_missingness_ARI": "Missingness ARI",
    "harmonized_missingness_SD": "Missingness SD",
    "utilization_ARI": "Utilization-only ARI",
    "decision": "Exploratory decision",
    "reasons": "Reason"
})

assert table2["Candidate"].is_unique
assert len(table2) == 5

# Validate completeness of primary metrics
primary_metrics = [
    "Seed ARI",
    "Subsample ARI",
    "Missingness ARI"
]

assert table2[primary_metrics].notna().all().all(), (
    "Missing primary reliability metrics"
)

# Clinical-only solutions have no utilization-only comparison
table2["Utilization-only ARI"] = (
    table2["Utilization-only ARI"]
    .fillna(np.nan)
)

display(table2.round(3))

table2.to_csv(
    RESULTS / "step28b_table2_reliability.csv",
    index=False
)

print("\nSTEP 28B-3: COMPLETE")


In [ ]:

import pandas as pd

# Load results
internal = pd.read_csv(
    RESULTS / "step15_kmeans_internal_metrics.csv"
)

selection = pd.read_csv(
    RESULTS / "step25i_reliability_selection.csv"
)

# Correct representation mapping
rep_map = {
    "R1_clinical": "R1",
    "R2_clinical_utilization": "R2",
    "R3_clinical_documentation": "R3"
}

internal["rep_short"] = internal["representation"].map(rep_map)

# Validate mapping
assert internal["rep_short"].notna().all(), (
    "Unknown representation found"
)

# Create matching solution identifiers
internal["solution"] = (
    internal["rep_short"]
    + "_K"
    + internal["k"].astype(int).astype(str)
)

assert internal["solution"].is_unique
assert selection["solution"].is_unique

# Join reliability and clustering metrics
final_table = selection.merge(
    internal[
        [
            "solution",
            "silhouette",
            "min_cluster_size",
            "max_cluster_size"
        ]
    ],
    on="solution",
    how="left",
    validate="one_to_one"
)

assert len(final_table) == 5
assert final_table["silhouette"].notna().all()

# Select manuscript columns
final_table = final_table[
    [
        "solution",
        "silhouette",
        "min_cluster_size",
        "max_cluster_size",
        "seed_ARI",
        "subsample_ARI",
        "harmonized_missingness_ARI",
        "harmonized_missingness_SD",
        "utilization_ARI",
        "decision",
        "reasons"
    ]
]

final_table = final_table.rename(columns={
    "solution": "Candidate",
    "silhouette": "Silhouette",
    "min_cluster_size": "Smallest cluster",
    "max_cluster_size": "Largest cluster",
    "seed_ARI": "Seed ARI",
    "subsample_ARI": "Subsample ARI",
    "harmonized_missingness_ARI": "Missingness ARI",
    "harmonized_missingness_SD": "Missingness SD",
    "utilization_ARI": "Utilization-only ARI",
    "decision": "Exploratory decision",
    "reasons": "Flag reason"
})

# Display results
display(final_table.round(3))

# Save full-precision results
final_table.to_csv(
    RESULTS / "step28b_table2_final.csv",
    index=False
)

print("\nSTEP 28B-4: PASS")
print("Candidates:", len(final_table))
print("Saved: step28b_table2_final.csv")


In [ ]:

import pandas as pd
from pathlib import Path

RESULTS = Path(
    r"C:\Projects\healthcare-cluster-reliability\results"
)

files = {
    "Held-out readmission statistics":
        "step28_heldout_readmission_statistics.csv",
    "Held-out missingness":
        "step26b_heldout_missingness_summary.csv",
    "Held-out representation dominance":
        "step26b_heldout_representation_dominance.csv"
}

for label, filename in files.items():

    df = pd.read_csv(RESULTS / filename)

    print("\n" + "=" * 65)
    print(label.upper())
    print("File:", filename)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    display(df.head(12))

print("\nSTEP 28B-5A: COMPLETE")


In [ ]:

import pandas as pd

files = {
    "Readmission statistics":
        "step28_heldout_readmission_statistics.csv",
    "Held-out missingness":
        "step26b_heldout_missingness_summary.csv"
}

for label, filename in files.items():
    df = pd.read_csv(RESULTS / filename)

    print("\n" + "=" * 60)
    print(label)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.to_string(index=False))

print("\nSTEP 28B-5B: COMPLETE")


In [ ]:

import pandas as pd
import numpy as np

# --------------------------------------------------
# Step 28B-6: Final held-out manuscript table
# --------------------------------------------------

stats = pd.read_csv(
    RESULTS / "step28_heldout_readmission_statistics.csv"
)

missing = pd.read_csv(
    RESULTS / "step26b_heldout_missingness_summary.csv"
)

dominance = pd.read_csv(
    RESULTS / "step26b_heldout_representation_dominance.csv"
)

assignments = pd.read_parquet(
    RESULTS / "step26_heldout_assignments.parquet"
)

# Validate patient cohort
assert len(assignments) == 61518
assert assignments["patient_nbr"].is_unique

# --------------------------------------------------
# Summarize cluster sizes
# --------------------------------------------------

cluster_sizes = []

for rep in ["R1", "R2"]:

    counts = assignments[
        f"cluster_{rep}"
    ].value_counts()

    assert set(counts.index) == {0, 1}

    cluster_sizes.append({
        "representation": rep,
        "cluster_0_n": int(counts[0]),
        "cluster_1_n": int(counts[1])
    })

cluster_sizes = pd.DataFrame(cluster_sizes)

# --------------------------------------------------
# Extract fixed-model masking results
# --------------------------------------------------

masking = missing[
    missing["missing_rate"].isin([0.20, 0.30])
].copy()

masking["masking_level"] = (
    masking["missing_rate"]
    .mul(100)
    .astype(int)
    .astype(str)
)

masking_wide = masking.pivot(
    index="representation",
    columns="masking_level",
    values="mean_ARI"
)

masking_wide = masking_wide.rename(columns={
    "20": "ARI_20pct_masking",
    "30": "ARI_30pct_masking"
}).reset_index()

# --------------------------------------------------
# Representation dominance
# --------------------------------------------------

dominance_small = dominance[
    ["representation", "heldout_ARI"]
].rename(columns={
    "heldout_ARI": "utilization_only_ARI"
})

# --------------------------------------------------
# Merge
# --------------------------------------------------

table3 = (
    stats
    .merge(
        cluster_sizes,
        on="representation",
        validate="one_to_one"
    )
    .merge(
        masking_wide,
        on="representation",
        validate="one_to_one"
    )
    .merge(
        dominance_small,
        on="representation",
        how="left",
        validate="one_to_one"
    )
)

assert len(table3) == 2

assert (
    table3["cluster_0_n"]
    + table3["cluster_1_n"]
    == 61518
).all()

assert (
    table3["ci_lower"]
    <= table3["risk_difference"]
).all()

assert (
    table3["risk_difference"]
    <= table3["ci_upper"]
).all()

# Rename columns for manuscript
table3 = table3.rename(columns={
    "representation": "Representation",
    "cluster_0_n": "Cluster 0 n",
    "cluster_1_n": "Cluster 1 n",
    "cluster_0_rate": "Cluster 0 readmission",
    "cluster_1_rate": "Cluster 1 readmission",
    "risk_difference": "Risk difference",
    "ci_lower": "95% CI lower",
    "ci_upper": "95% CI upper",
    "chi2_p": "Chi-square p",
    "ARI_20pct_masking": "ARI at 20% masking",
    "ARI_30pct_masking": "ARI at 30% masking",
    "utilization_only_ARI": "Utilization-only ARI"
})

# Order columns
table3 = table3[
    [
        "Representation",
        "Cluster 0 n",
        "Cluster 1 n",
        "Cluster 0 readmission",
        "Cluster 1 readmission",
        "Risk difference",
        "95% CI lower",
        "95% CI upper",
        "Chi-square p",
        "ARI at 20% masking",
        "ARI at 30% masking",
        "Utilization-only ARI"
    ]
]

display(table3.round(4))

# Save full-precision evidence
table3.to_csv(
    RESULTS / "step28b_table3_heldout_final.csv",
    index=False
)

print("\nSTEP 28B-6: PASS")
print("Saved: step28b_table3_heldout_final.csv")


In [ ]:

import pandas as pd

table3 = pd.read_csv(
    RESULTS / "step28b_table3_heldout_final.csv"
)

dominance = pd.read_csv(
    RESULTS / "step26b_heldout_representation_dominance.csv"
)

# Convert R2_K2 to R2 for the table join
dominance["Representation"] = (
    dominance["representation"]
    .str.extract(r"^(R\d+)_K\d+$")[0]
)

assert dominance["Representation"].notna().all()
assert dominance["Representation"].is_unique

# Remove incorrectly joined column
table3 = table3.drop(
    columns=["Utilization-only ARI"]
)

# Join corrected representation dominance
table3 = table3.merge(
    dominance[
        ["Representation", "heldout_ARI"]
    ],
    on="Representation",
    how="left",
    validate="one_to_one"
)

table3 = table3.rename(columns={
    "heldout_ARI": "Utilization-only ARI"
})

# Validation
assert len(table3) == 2

r2_ari = table3.loc[
    table3["Representation"] == "R2",
    "Utilization-only ARI"
].iloc[0]

assert r2_ari == 1.0

# R1 has no utilization-only comparison
assert pd.isna(
    table3.loc[
        table3["Representation"] == "R1",
        "Utilization-only ARI"
    ].iloc[0]
)

# Save corrected version
table3.to_csv(
    RESULTS / "step28b_table3_heldout_final.csv",
    index=False
)

display(table3.round(4))

print("\nSTEP 28B-7: PASS")
print("R2 utilization-only ARI:", r2_ari)
print("Corrected Table 3 saved.")


In [ ]:

import pandas as pd
import numpy as np

# --------------------------------------------------
# Step 29A: Patient separation and outcome audit
# --------------------------------------------------

development = pd.read_parquet(
    DATA / "development_R1_10k.parquet"
)

heldout = pd.read_parquet(
    RESULTS / "step26_heldout_assignments.parquet"
)

table3 = pd.read_csv(
    RESULTS / "step28b_table3_heldout_final.csv"
)

# 1. Patient uniqueness
assert development["patient_nbr"].is_unique
assert heldout["patient_nbr"].is_unique

# 2. No patient overlap
overlap = set(development["patient_nbr"]) & set(
    heldout["patient_nbr"]
)

assert len(overlap) == 0

# 3. Expected cohort sizes
assert len(development) == 10000
assert len(heldout) == 61518

# 4. Outcome completeness
assert heldout["readmit_30d"].notna().all()

# 5. Recalculate readmission statistics
audit_rows = []

for rep in ["R1", "R2"]:

    cluster_col = f"cluster_{rep}"

    grouped = heldout.groupby(cluster_col)["readmit_30d"].agg(
        ["size", "sum", "mean"]
    )

    reported = table3.loc[
        table3["Representation"] == rep
    ].iloc[0]

    for cluster in [0, 1]:

        n = int(grouped.loc[cluster, "size"])
        rate = float(grouped.loc[cluster, "mean"])

        expected_n = reported[f"Cluster {cluster} n"]
        expected_rate = reported[
            f"Cluster {cluster} readmission"
        ]

        assert n == expected_n
        assert np.isclose(
            rate, expected_rate, atol=1e-6
        )

        audit_rows.append({
            "representation": rep,
            "cluster": cluster,
            "patients": n,
            "readmissions": int(
                grouped.loc[cluster, "sum"]
            ),
            "readmission_rate": rate
        })

audit_df = pd.DataFrame(audit_rows)

print("PATIENT SEPARATION")
print("-" * 50)
print("Development patients:", len(development))
print("Held-out patients:", len(heldout))
print("Overlapping patients:", len(overlap))
print("Missing held-out outcomes:",
      heldout["readmit_30d"].isna().sum())

print("\nREADMISSION RECONCILIATION")
display(audit_df)

audit_df.to_csv(
    RESULTS / "step29a_patient_outcome_audit.csv",
    index=False
)

print("\nSTEP 29A: PASS")


In [ ]:

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, norm

# --------------------------------------------------
# Step 29B: Independent statistical verification
# --------------------------------------------------

audit = pd.read_csv(
    RESULTS / "step29a_patient_outcome_audit.csv"
)

reported = pd.read_csv(
    RESULTS / "step28b_table3_heldout_final.csv"
)

rows = []

for rep in ["R1", "R2"]:

    sub = (
        audit[audit["representation"] == rep]
        .set_index("cluster")
    )

    n0 = int(sub.loc[0, "patients"])
    n1 = int(sub.loc[1, "patients"])

    y0 = int(sub.loc[0, "readmissions"])
    y1 = int(sub.loc[1, "readmissions"])

    p0 = y0 / n0
    p1 = y1 / n1

    # Unadjusted absolute risk difference
    rd = p1 - p0

    # Unpooled Wald 95% confidence interval
    se = np.sqrt(
        p0 * (1 - p0) / n0
        + p1 * (1 - p1) / n1
    )

    z = norm.ppf(0.975)

    ci_lower = rd - z * se
    ci_upper = rd + z * se

    # Pearson chi-square, no Yates correction
    contingency = np.array([
        [y0, n0 - y0],
        [y1, n1 - y1]
    ])

    chi2, p_value, dof, expected = (
        chi2_contingency(
            contingency,
            correction=False
        )
    )

    ref = reported.loc[
        reported["Representation"] == rep
    ].iloc[0]

    rows.append({
        "representation": rep,
        "risk_difference": rd,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "chi2_p": p_value,
        "minimum_expected": expected.min(),
        "rd_matches": np.isclose(
            rd, ref["Risk difference"], atol=1e-6
        ),
        "ci_matches": (
            np.isclose(
                ci_lower,
                ref["95% CI lower"],
                atol=1e-6
            )
            and np.isclose(
                ci_upper,
                ref["95% CI upper"],
                atol=1e-6
            )
        ),
        "p_matches": np.isclose(
            p_value,
            ref["Chi-square p"],
            atol=1e-6
        )
    })

verification = pd.DataFrame(rows)

display(verification.round(6))

assert verification[
    ["rd_matches", "ci_matches", "p_matches"]
].all().all(), "Statistical discrepancy detected"

verification.to_csv(
    RESULTS / "step29b_statistical_verification.csv",
    index=False
)

print("\nSTEP 29B: PASS")


In [ ]:

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# --------------------------------------------------
# Step 29C-1: Missing-category encoding audit
# --------------------------------------------------

dev = pd.read_parquet(
    DATA / "development_R1_10k.parquet"
)

CLIN = [
    "age",
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

print("TRAINING MISSING-VALUE AUDIT")
print("-" * 55)

for col in CLIN:
    s = dev[col].astype("string")

    print(
        col,
        "| null:",
        int(s.isna().sum()),
        "| literal missing:",
        int(s.eq("missing").sum()),
        "| distinct:",
        int(s.nunique(dropna=True))
    )

# Match the recent experimental categorical pipeline
categorical = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

categorical.fit(dev[CLIN])

encoder = categorical.named_steps["encoder"]

print("\nFITTED ENCODER CATEGORY AUDIT")
print("-" * 55)

for col, categories in zip(CLIN, encoder.categories_):
    has_missing = "missing" in set(
        str(x) for x in categories
    )

    print(
        col,
        "| explicit missing category:",
        has_missing,
        "| category count:",
        len(categories)
    )

# Compare a real training row against a masked row
sample = dev[CLIN].iloc[[0]].copy()
masked = sample.copy()

masked["diag1_family"] = "missing"

original_encoded = categorical.transform(sample)
masked_encoded = categorical.transform(masked)

if hasattr(original_encoded, "toarray"):
    original_encoded = original_encoded.toarray()

if hasattr(masked_encoded, "toarray"):
    masked_encoded = masked_encoded.toarray()

print("\nENCODING CHANGE")
print("-" * 55)

print(
    "Changed encoded dimensions:",
    int(np.count_nonzero(
        original_encoded != masked_encoded
    ))
)

print(
    "Encoded feature count:",
    masked_encoded.shape[1]
)

print("\nSTEP 29C-1: COMPLETE")


In [ ]:

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import adjusted_rand_score

# --------------------------------------------------
# Step 29C-2: Diagnosis-position sensitivity
# --------------------------------------------------

dev = pd.read_parquet(
    DATA / "development_R1_10k.parquet"
)

CLIN = [
    "age",
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

categorical = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

X = categorical.fit_transform(dev[CLIN])

model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

baseline_labels = model.fit_predict(X)

rng = np.random.default_rng(42)

rows = []

for col in [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]:

    for rate in [0.05, 0.10, 0.20, 0.30]:

        for repeat in range(20):

            perturbed = dev[CLIN].copy()

            selected = rng.choice(
                len(perturbed),
                size=round(rate * len(perturbed)),
                replace=False
            )

            perturbed.iloc[
                selected,
                perturbed.columns.get_loc(col)
            ] = "missing"

            X_perturbed = categorical.transform(
                perturbed
            )

            labels = model.predict(X_perturbed)

            rows.append({
                "diagnosis_position": col,
                "missing_rate": rate,
                "repeat": repeat,
                "ARI": adjusted_rand_score(
                    baseline_labels,
                    labels
                ),
                "fraction_changed": np.mean(
                    baseline_labels != labels
                )
            })

position_runs = pd.DataFrame(rows)

position_summary = (
    position_runs
    .groupby(
        ["diagnosis_position", "missing_rate"]
    )
    .agg(
        mean_ARI=("ARI", "mean"),
        std_ARI=("ARI", "std"),
        mean_fraction_changed=(
            "fraction_changed", "mean"
        )
    )
    .reset_index()
)

display(position_summary.round(4))

position_runs.to_csv(
    RESULTS / "step29c_position_masking_runs.csv",
    index=False
)

position_summary.to_csv(
    RESULTS / "step29c_position_masking_summary.csv",
    index=False
)

print("\nSTEP 29C-2: PASS")


In [ ]:

import pandas as pd
import numpy as np
from sklearn.metrics import adjusted_rand_score

# --------------------------------------------------
# Step 29C-3: Masking implementation audit
# --------------------------------------------------

rng = np.random.default_rng(2026)

diagnosis_cols = [
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

rate = 0.20
n_repeats = 20

rows = []

for repeat in range(n_repeats):

    perturbed = dev[CLIN].copy()

    for col in diagnosis_cols:

        selected = rng.choice(
            len(perturbed),
            size=round(rate * len(perturbed)),
            replace=False
        )

        before = perturbed[col].copy()

        perturbed.iloc[
            selected,
            perturbed.columns.get_loc(col)
        ] = "missing"

        actual_changed = int(
            (before != perturbed[col]).sum()
        )

        rows.append({
            "repeat": repeat,
            "diagnosis_position": col,
            "selected_for_masking": len(selected),
            "actual_values_changed": actual_changed
        })

mask_audit = pd.DataFrame(rows)

summary = (
    mask_audit
    .groupby("diagnosis_position")
    .agg(
        mean_selected=("selected_for_masking", "mean"),
        mean_changed=("actual_values_changed", "mean"),
        min_changed=("actual_values_changed", "min"),
        max_changed=("actual_values_changed", "max")
    )
    .reset_index()
)

display(summary.round(2))

# Check that all diagnosis positions are being changed
assert (summary["min_changed"] > 0).all()

mask_audit.to_csv(
    RESULTS / "step29c_masking_implementation_audit.csv",
    index=False
)

print("\nSTEP 29C-3: PASS")



In [ ]:

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# --------------------------------------------------
# Step 29D-1: R1 cluster centroid separation
# --------------------------------------------------

dev = pd.read_parquet(
    DATA / "development_R1_10k.parquet"
)

CLIN = [
    "age",
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

categorical = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

X = categorical.fit_transform(dev[CLIN])

model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

model.fit(X)

encoder = categorical.named_steps["encoder"]

centroids = model.cluster_centers_

# Identify encoded feature blocks
offset = 0
rows = []

for col, categories in zip(
    CLIN,
    encoder.categories_
):

    width = len(categories)

    block_0 = centroids[0, offset:offset + width]
    block_1 = centroids[1, offset:offset + width]

    distance = np.linalg.norm(
        block_0 - block_1
    )

    rows.append({
        "feature_block": col,
        "encoded_dimensions": width,
        "centroid_distance": distance,
        "squared_distance": distance ** 2
    })

    offset += width

assert offset == X.shape[1]

centroid_audit = pd.DataFrame(rows)

centroid_audit["share_of_squared_distance"] = (
    centroid_audit["squared_distance"]
    / centroid_audit["squared_distance"].sum()
)

display(
    centroid_audit.sort_values(
        "share_of_squared_distance",
        ascending=False
    ).round(4)
)

centroid_audit.to_csv(
    RESULTS / "step29d_centroid_block_separation.csv",
    index=False
)

print("\nSTEP 29D-1: COMPLETE")


In [ ]:

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

# --------------------------------------------------
# Step 29D-2: Centroid dominance across seeds
# --------------------------------------------------

# Reuse X and encoder from Step 29D-1
assert X.shape[0] == 10000

feature_blocks = {}
offset = 0

for col, categories in zip(
    CLIN,
    encoder.categories_
):
    width = len(categories)
    feature_blocks[col] = slice(
        offset,
        offset + width
    )
    offset += width

assert offset == X.shape[1]

rows = []

for seed in range(30):

    km = KMeans(
        n_clusters=2,
        random_state=seed,
        n_init=20
    )

    km.fit(X)

    differences = (
        km.cluster_centers_[0]
        - km.cluster_centers_[1]
    )

    total_squared = np.sum(
        differences ** 2
    )

    for feature, block in feature_blocks.items():

        squared = np.sum(
            differences[block] ** 2
        )

        rows.append({
            "seed": seed,
            "feature_block": feature,
            "share_of_squared_distance": (
                squared / total_squared
            )
        })

seed_dominance = pd.DataFrame(rows)

summary = (
    seed_dominance
    .groupby("feature_block")
    .agg(
        mean_share=(
            "share_of_squared_distance",
            "mean"
        ),
        std_share=(
            "share_of_squared_distance",
            "std"
        ),
        min_share=(
            "share_of_squared_distance",
            "min"
        ),
        max_share=(
            "share_of_squared_distance",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        "mean_share",
        ascending=False
    )
)

display(summary.round(4))

seed_dominance.to_csv(
    RESULTS / "step29d_centroid_dominance_seed_runs.csv",
    index=False
)

summary.to_csv(
    RESULTS / "step29d_centroid_dominance_seed_summary.csv",
    index=False
)

print("\nSTEP 29D-2: COMPLETE")



In [ ]:

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

# --------------------------------------------------
# Step 29D-3: Feature concentration across K
# --------------------------------------------------

# Reuse X, CLIN, encoder, and feature_blocks
# from Step 29D-2.

assert X.shape[0] == 10000

rows = []

for k in [2, 5, 8]:

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = km.fit_predict(X)

    counts = np.bincount(
        labels,
        minlength=k
    )

    centroids = km.cluster_centers_

    # Overall weighted centroid
    overall = np.average(
        centroids,
        axis=0,
        weights=counts
    )

    # Weighted between-cluster dispersion
    squared_deviation = (
        centroids - overall
    ) ** 2

    weighted_deviation = (
        squared_deviation
        * counts[:, None]
    )

    total = weighted_deviation.sum()

    for feature, block in feature_blocks.items():

        contribution = (
            weighted_deviation[:, block].sum()
        )

        rows.append({
            "k": k,
            "feature_block": feature,
            "share_between_cluster_dispersion": (
                contribution / total
            )
        })

k_dominance = pd.DataFrame(rows)

comparison = (
    k_dominance
    .pivot(
        index="feature_block",
        columns="k",
        values="share_between_cluster_dispersion"
    )
    .rename(
        columns={
            2: "K2",
            5: "K5",
            8: "K8"
        }
    )
)

display(comparison.round(4))

k_dominance.to_csv(
    RESULTS / "step29d_feature_concentration_across_k.csv",
    index=False
)

print("\nSTEP 29D-3: COMPLETE")


In [ ]:

from pathlib import Path
import pandas as pd
import hashlib
import json
from datetime import datetime, timezone

# --------------------------------------------------
# STEP 30A: Research evidence manifest
# --------------------------------------------------

PROJECT = Path(
    r"C:\Projects\healthcare-cluster-reliability"
)

DATA = PROJECT / "data" / "processed"
RESULTS = PROJECT / "results"
FIGURES = PROJECT / "figures"

RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# Core evidence required for manuscript figures/tables
required_files = [
    RESULTS / "step28b_table2_final.csv",
    RESULTS / "step28b_table3_heldout_final.csv",
    RESULTS / "step26b_heldout_missingness_runs.csv",
    RESULTS / "step26b_heldout_missingness_summary.csv",
    RESULTS / "step26b_heldout_representation_dominance.csv",
    RESULTS / "step29a_patient_outcome_audit.csv",
    RESULTS / "step29b_statistical_verification.csv",
    RESULTS / "step29c_position_masking_summary.csv",
    RESULTS / "step29c_masking_implementation_audit.csv",
    RESULTS / "step29d_centroid_block_separation.csv",
    RESULTS / "step29d_centroid_dominance_seed_summary.csv",
    RESULTS / "step29d_feature_concentration_across_k.csv",
    RESULTS / "step27f_harmonization_audit.csv",
    RESULTS / "step26_heldout_assignments.parquet",
    DATA / "development_R1_10k.parquet",
    DATA / "development_R2_10k.parquet",
    DATA / "development_R3_10k.parquet",
]

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

rows = []

for path in required_files:
    exists = path.is_file()

    rows.append({
        "file": path.name,
        "relative_path": str(
            path.relative_to(PROJECT)
        ),
        "exists": exists,
        "size_bytes": (
            path.stat().st_size if exists else None
        ),
        "sha256": (
            sha256_file(path) if exists else None
        )
    })

manifest = pd.DataFrame(rows)

manifest.to_csv(
    RESULTS / "step30a_evidence_manifest.csv",
    index=False
)

summary = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "project": PROJECT.name,
    "total_expected": len(manifest),
    "total_present": int(manifest["exists"].sum()),
    "total_missing": int((~manifest["exists"]).sum()),
    "manifest": manifest.to_dict(
        orient="records"
    )
}

with open(
    RESULTS / "step30a_evidence_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

print("EVIDENCE FREEZE AUDIT")
print("-" * 55)

print("Expected files:", summary["total_expected"])
print("Present files:", summary["total_present"])
print("Missing files:", summary["total_missing"])

if summary["total_missing"]:
    print("\nMISSING FILES:")
    display(
        manifest.loc[
            ~manifest["exists"],
            ["file", "relative_path"]
        ]
    )
else:
    print("\nAll required evidence files found.")

print("\nManifest saved to:")
print(RESULTS / "step30a_evidence_manifest.csv")
print(RESULTS / "step30a_evidence_manifest.json")

print("\nSTEP 30A:", (
    "PASS" if summary["total_missing"] == 0
    else "REVIEW REQUIRED"
))


In [ ]:

import pandas as pd
import numpy as np

# --------------------------------------------------
# STEP 30B: Final evidence consistency audit
# --------------------------------------------------

table2 = pd.read_csv(
    RESULTS / "step28b_table2_final.csv"
)

table3 = pd.read_csv(
    RESULTS / "step28b_table3_heldout_final.csv"
)

missingness = pd.read_csv(
    RESULTS / "step26b_heldout_missingness_summary.csv"
)

dominance = pd.read_csv(
    RESULTS / "step26b_heldout_representation_dominance.csv"
)

stats = pd.read_csv(
    RESULTS / "step29b_statistical_verification.csv"
)

heldout = pd.read_parquet(
    RESULTS / "step26_heldout_assignments.parquet"
)

checks = []

def record_check(name, passed, details):
    checks.append({
        "check": name,
        "passed": bool(passed),
        "details": str(details)
    })

# 1. Table 2 candidate count
record_check(
    "Five candidate solutions",
    len(table2) == 5,
    f"Found {len(table2)} candidates"
)

# 2. All candidates flagged
decision_cols = [
    c for c in table2.columns
    if "decision" in c.lower()
]

if len(decision_cols) == 1:
    decisions = (
        table2[decision_cols[0]]
        .astype(str)
        .str.upper()
    )
    flagged = decisions.str.contains(
        "FLAGGED", regex=False
    ).all()
else:
    flagged = False

record_check(
    "No candidate declared reliable",
    flagged,
    f"Decision columns: {decision_cols}"
)

# 3. Held-out cluster counts
for rep in ["R1", "R2"]:

    row = table3.loc[
        table3["Representation"] == rep
    ].iloc[0]

    expected_n = (
        int(row["Cluster 0 n"])
        + int(row["Cluster 1 n"])
    )

    record_check(
        f"{rep} held-out count",
        expected_n == len(heldout),
        f"{expected_n} vs {len(heldout)}"
    )

# 4. Independently verified statistics
for rep in ["R1", "R2"]:

    row = stats.loc[
        stats["representation"] == rep
    ].iloc[0]

    passed = all([
        bool(row["rd_matches"]),
        bool(row["ci_matches"]),
        bool(row["p_matches"])
    ])

    record_check(
        f"{rep} statistical verification",
        passed,
        "Risk difference, CI and p-value"
    )

# 5. Held-out masking agreement
# Detect relevant column names before comparison
print("MISSINGNESS SUMMARY COLUMNS:")
print(missingness.columns.tolist())

print("\nDOMINANCE COLUMNS:")
print(dominance.columns.tolist())

# 6. Expected masking coverage
if {
    "representation",
    "missing_rate"
}.issubset(missingness.columns):

    expected = {
        ("R1", 0.05), ("R1", 0.10),
        ("R1", 0.20), ("R1", 0.30),
        ("R2", 0.05), ("R2", 0.10),
        ("R2", 0.20), ("R2", 0.30)
    }

    observed = set(
        zip(
            missingness["representation"],
            missingness["missing_rate"].round(2)
        )
    )

    record_check(
        "Missingness coverage",
        expected == observed,
        f"{len(observed)} representation/rate pairs"
    )

# 7. R2 utilization-only equivalence
r2_row = table3.loc[
    table3["Representation"] == "R2"
].iloc[0]

util_cols = [
    c for c in table3.columns
    if "utilization" in c.lower()
    and "ari" in c.lower()
]

if len(util_cols) == 1:
    util_ari = float(r2_row[util_cols[0]])
    util_pass = np.isclose(util_ari, 1.0)
else:
    util_ari = None
    util_pass = False

record_check(
    "R2 utilization-only equivalence",
    util_pass,
    f"ARI={util_ari}"
)

# Final report
audit = pd.DataFrame(checks)

display(audit)

audit.to_csv(
    RESULTS / "step30b_consistency_audit.csv",
    index=False
)

print("\nFINAL AUDIT")
print("-" * 55)
print("Checks:", len(audit))
print("Passed:", int(audit["passed"].sum()))
print("Failed:", int((~audit["passed"]).sum()))

if audit["passed"].all():
    print("\nSTEP 30B: PASS")
else:
    print("\nSTEP 30B: REVIEW REQUIRED")


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --------------------------------------------------
# STEP 31A: Figure 2 - Candidate reliability
# --------------------------------------------------

table = pd.read_csv(
    RESULTS / "step28b_table2_final.csv"
)

print("Available columns:")
print(table.columns.tolist())

# Locate columns by their reported meanings
def find_column(keywords):
    matches = [
        c for c in table.columns
        if all(
            word in c.lower()
            for word in keywords
        )
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Cannot uniquely identify {keywords}: {matches}"
        )
    return matches[0]

candidate_col = find_column(["candidate"])
sil_col = find_column(["silhouette"])
seed_col = find_column(["seed"])
sub_col = find_column(["subsample"])
miss_col = find_column(["missingness", "ari"])
decision_col = find_column(["decision"])

# Find utilization-only ARI; do not confuse with
# the missingness ARI column.
util_candidates = [
    c for c in table.columns
    if "util" in c.lower()
    and "ari" in c.lower()
]

if len(util_candidates) != 1:
    raise ValueError(
        "Utilization ARI column ambiguous: "
        + str(util_candidates)
    )

util_col = util_candidates[0]

names = table[candidate_col].astype(str).tolist()
x = np.arange(len(names))

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 5.2),
    gridspec_kw={
        "width_ratios": [1, 1.5, 1.2]
    }
)

# Panel A: Internal quality
axes[0].bar(
    x,
    table[sil_col].astype(float),
    color="#4678A8"
)
axes[0].set_xticks(x, names, rotation=45, ha="right")
axes[0].set_ylim(0, 0.5)
axes[0].set_ylabel("Silhouette score")
axes[0].set_title("A. Internal cluster quality")
axes[0].grid(axis="y", alpha=0.2)

# Panel B: Reliability dimensions
metrics = [
    (seed_col, "Seed stability", "#2675C9"),
    (sub_col, "Subsample stability", "#36A59A"),
    (miss_col, "Refitted missingness", "#D08A35")
]

bar_width = 0.25

for j, (col, label, color) in enumerate(metrics):
    axes[1].bar(
        x + (j - 1) * bar_width,
        table[col].astype(float),
        width=bar_width,
        label=label,
        color=color
    )

axes[1].set_xticks(x, names, rotation=45, ha="right")
axes[1].set_ylim(0, 1.08)
axes[1].set_ylabel("Adjusted Rand index")
axes[1].set_title("B. Reliability under perturbation")
axes[1].legend(
    fontsize=8,
    frameon=False,
    loc="upper center"
)
axes[1].grid(axis="y", alpha=0.2)

# Panel C: Utilization equivalence
util_values = pd.to_numeric(
    table[util_col],
    errors="coerce"
)

colors = [
    "#B9C4CE" if pd.isna(v)
    else "#B45F5F"
    for v in util_values
]

axes[2].bar(
    x,
    util_values.fillna(0),
    color=colors
)

axes[2].set_xticks(x, names, rotation=45, ha="right")
axes[2].set_ylim(0, 1.1)
axes[2].set_ylabel("ARI vs utilization-only")
axes[2].set_title("C. Representation dominance")
axes[2].grid(axis="y", alpha=0.2)

for i, v in enumerate(util_values):
    if pd.isna(v):
        axes[2].text(
            i, 0.05, "N/A",
            ha="center",
            va="bottom",
            fontsize=9
        )
    else:
        axes[2].text(
            i, min(v + 0.04, 1.06),
            f"{v:.2f}",
            ha="center",
            fontsize=9
        )

fig.suptitle(
    "Candidate clustering quality and reliability",
    fontsize=15,
    fontweight="bold",
    y=1.02
)

fig.text(
    0.5, -0.02,
    "Source: frozen Step 28B Table 2. "
    "Missingness ARI uses model refitting; "
    "N/A indicates a comparison not evaluated. "
    "All five candidates were flagged under "
    "exploratory reliability criteria.",
    ha="center",
    va="top",
    fontsize=8.5,
    wrap=True
)

plt.tight_layout()

png = FIGURES / "figure2_candidate_reliability.png"
pdf = FIGURES / "figure2_candidate_reliability.pdf"

fig.savefig(
    png,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    pdf,
    bbox_inches="tight"
)

plt.show()

print("\nFIGURE FILES")
print(png)
print(pdf)

print("\nSTEP 31A: COMPLETE")


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# STEP 31B: Refined publication Figure 2
# --------------------------------------------------

df = pd.read_csv(
    RESULTS / "step28b_table2_final.csv"
)

names = df["Candidate"].astype(str).tolist()
x = np.arange(len(names))

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 6),
    gridspec_kw={
        "width_ratios": [1, 1.6, 1.1]
    }
)

# A: Internal quality
axes[0].bar(
    x, df["Silhouette"],
    color="#4678A8",
    width=0.65
)

axes[0].set(
    title="A. Internal cluster quality",
    ylabel="Silhouette score",
    ylim=(0, 0.5)
)

# B: Perturbation reliability
width = 0.24

series = [
    ("Seed ARI", "Seed stability", "#2675C9"),
    ("Subsample ARI", "Subsample stability", "#36A59A"),
    ("Missingness ARI", "Refitted missingness", "#D08A35")
]

for j, (col, label, color) in enumerate(series):
    axes[1].bar(
        x + (j - 1) * width,
        df[col],
        width=width,
        color=color,
        label=label
    )

axes[1].set(
    title="B. Perturbation reliability",
    ylabel="Adjusted Rand index",
    ylim=(0, 1.1)
)

axes[1].legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=3,
    frameon=False,
    fontsize=8
)

# C: Representation dominance
util = pd.to_numeric(
    df["Utilization-only ARI"],
    errors="coerce"
)

axes[2].bar(
    x,
    util.fillna(0),
    color="#B45F5F",
    width=0.65
)

axes[2].set(
    title="C. Utilization-only equivalence",
    ylabel="ARI vs utilization-only",
    ylim=(0, 1.12)
)

for i, v in enumerate(util):
    axes[2].text(
        i,
        0.04 if pd.isna(v) else v + 0.035,
        "N/A" if pd.isna(v) else f"{v:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# Shared formatting
for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(
        names,
        rotation=40,
        ha="right"
    )
    ax.grid(
        axis="y",
        alpha=0.18
    )
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "Cluster quality does not establish phenotype reliability",
    fontsize=15,
    fontweight="bold",
    y=0.99
)

fig.text(
    0.5, 0.035,
    "Panel B: missingness robustness uses model refitting. "
    "Panel C: ARI = 1 indicates an identical utilization-only partition; "
    "N/A = not evaluated.",
    ha="center",
    fontsize=9
)

fig.text(
    0.5, 0.008,
    "Source: Step 28B frozen Table 2; "
    "10,000 development patients. "
    "All five candidates flagged under exploratory criteria.",
    ha="center",
    fontsize=8,
    color="#555555"
)

fig.subplots_adjust(
    left=0.06,
    right=0.98,
    bottom=0.20,
    top=0.70,
    wspace=0.35
)
png = FIGURES / "figure2_candidate_reliability_final.png"
pdf = FIGURES / "figure2_candidate_reliability_final.pdf"

fig.savefig(
    png,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    pdf,
    bbox_inches="tight"
)

plt.show()

print("Saved:", png)
print("Saved:", pdf)
print("STEP 31B: COMPLETE")


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# STEP 31C: Figure 3 - Held-out missingness
# --------------------------------------------------

df = pd.read_csv(
    RESULTS / "step26b_heldout_missingness_summary.csv"
)

required = {
    "representation",
    "missing_rate",
    "mean_ARI",
    "std_ARI",
    "mean_fraction_changed"
}

assert required.issubset(df.columns)
assert len(df) == 8

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 4.8)
)

series = [
    ("R1", "Clinical only", "#2675C9"),
    ("R2", "Clinical + utilization", "#C47D35")
]

for rep, label, color in series:

    sub = (
        df[df["representation"] == rep]
        .sort_values("missing_rate")
    )

    x = sub["missing_rate"].to_numpy() * 100

    # Panel A: Assignment agreement
    axes[0].errorbar(
        x,
        sub["mean_ARI"],
        yerr=sub["std_ARI"],
        marker="o",
        linewidth=2,
        capsize=3,
        color=color,
        label=label
    )

    # Panel B: Fraction of assignments changed
    axes[1].plot(
        x,
        sub["mean_fraction_changed"] * 100,
        marker="o",
        linewidth=2,
        color=color,
        label=label
    )

axes[0].set(
    title="A. Cluster assignment agreement",
    xlabel="Diagnosis masking rate (%)",
    ylabel="Adjusted Rand index",
    ylim=(0.55, 1.05)
)

axes[1].set(
    title="B. Patients changing cluster assignment",
    xlabel="Diagnosis masking rate (%)",
    ylabel="Mean assignments changed (%)",
    ylim=(-0.5, 11)
)

for ax in axes:
    ax.set_xticks([5, 10, 20, 30])
    ax.grid(axis="y", alpha=0.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.87),
    ncol=2,
    frameon=False
)

fig.suptitle(
    "Held-out assignment robustness under diagnosis masking",
    fontsize=14,
    fontweight="bold",
    y=0.98
)

fig.text(
    0.5, 0.035,
    "Source: Step 26B held-out missingness summary; "
    "61,518 patient-disjoint held-out records. "
    "Development-fitted models held fixed; "
    "20 masking repetitions per rate.",
    ha="center",
    fontsize=8.5
)

fig.subplots_adjust(
    left=0.09,
    right=0.97,
    bottom=0.19,
    top=0.70,
    wspace=0.30
)

png = FIGURES / "figure3_heldout_missingness.png"
pdf = FIGURES / "figure3_heldout_missingness.pdf"

fig.savefig(png, dpi=300, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")

plt.show()

print("Saved:", png)
print("Saved:", pdf)
print("STEP 31C: COMPLETE")


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# STEP 31D: Figure 4 - Feature concentration
# and representation dominance
# --------------------------------------------------

concentration = pd.read_csv(
    RESULTS / "step29d_feature_concentration_across_k.csv"
)

table2 = pd.read_csv(
    RESULTS / "step28b_table2_final.csv"
)

# Validate expected input columns
assert {
    "k",
    "feature_block",
    "share_between_cluster_dispersion"
}.issubset(concentration.columns)

assert {
    "Candidate",
    "Utilization-only ARI"
}.issubset(table2.columns)

feature_order = [
    "age",
    "diag1_family",
    "diag2_family",
    "diag3_family"
]

feature_labels = [
    "Age",
    "Diagnosis 1",
    "Diagnosis 2",
    "Diagnosis 3"
]

feature_colors = [
    "#4C78A8",
    "#59A14F",
    "#F28E2B",
    "#B279A2"
]

pivot = (
    concentration
    .pivot(
        index="k",
        columns="feature_block",
        values="share_between_cluster_dispersion"
    )
    .loc[[2, 5, 8], feature_order]
)

assert np.allclose(
    pivot.sum(axis=1).to_numpy(),
    1.0,
    atol=1e-3
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(12.5, 5),
    gridspec_kw={
        "width_ratios": [1.5, 1]
    }
)

# -----------------------------------
# Panel A: Feature concentration
# -----------------------------------

x = np.arange(len(pivot))
bottom = np.zeros(len(pivot))

for feature, label, color in zip(
    feature_order,
    feature_labels,
    feature_colors
):
    values = pivot[feature].to_numpy() * 100

    axes[0].bar(
        x,
        values,
        bottom=bottom,
        width=0.62,
        label=label,
        color=color
    )

    bottom += values

axes[0].set_xticks(
    x,
    ["K=2", "K=5", "K=8"]
)

axes[0].set(
    title="A. Feature concentration across K",
    ylabel="Between-cluster dispersion (%)",
    ylim=(0, 105)
)

axes[0].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.22),
    ncol=4,
    frameon=False,
    fontsize=8
)

# -----------------------------------
# Panel B: Utilization-only equivalence
# -----------------------------------

subset = table2[
    table2["Candidate"].isin([
        "R2_K2",
        "R3_K2"
    ])
].copy()

subset = subset.set_index(
    "Candidate"
).loc[["R2_K2", "R3_K2"]]

values = pd.to_numeric(
    subset["Utilization-only ARI"]
)

assert np.allclose(values, 1.0)

bars = axes[1].bar(
    ["R2 K=2", "R3 K=2"],
    values,
    color=["#B45F5F", "#9A6E9A"],
    width=0.55
)

for bar, value in zip(bars, values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.035,
        f"{value:.2f}",
        ha="center",
        fontsize=10
    )

axes[1].set(
    title="B. Utilization-only equivalence",
    ylabel="Adjusted Rand index",
    ylim=(0, 1.12)
)

# Shared formatting
for ax in axes:
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "Feature concentration and representation dominance",
    fontsize=14,
    fontweight="bold",
    y=0.99
)

fig.text(
    0.5, 0.04,
    "Panel A: development cohort (n=10,000), "
    "weighted between-cluster centroid dispersion. "
    "Panel B: development utilization-only comparison; "
    "ARI=1 indicates identical partitions.",
    ha="center",
    fontsize=8.5
)

fig.text(
    0.5, 0.012,
    "Sources: Step 29D feature concentration across K "
    "and frozen Step 28B Table 2.",
    ha="center",
    fontsize=8,
    color="#555555"
)

fig.subplots_adjust(
    left=0.08,
    right=0.97,
    bottom=0.18,
    top=0.70,
    wspace=0.30
)

png = FIGURES / "figure4_feature_concentration.png"
pdf = FIGURES / "figure4_feature_concentration.pdf"

fig.savefig(
    png,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    pdf,
    bbox_inches="tight"
)

plt.show()

print("Saved:", png)
print("Saved:", pdf)
print("STEP 31D: COMPLETE")


In [ ]:

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# --------------------------------------------------
# STEP 31E: Figure 1 - Reliability framework
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(13, 10))
ax.set_xlim(0, 13)
ax.set_ylim(0, 10)
ax.axis("off")

def box(x, y, w, h, title, subtitle="",
        face="#EEF3F8", edge="#6A879E"):
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.06,rounding_size=0.12",
        linewidth=1.2,
        edgecolor=edge,
        facecolor=face
    )
    ax.add_patch(patch)
    ax.text(
        x + w/2, y + h*0.64,
        title,
        ha="center", va="center",
        fontsize=10, fontweight="bold"
    )
    if subtitle:
        ax.text(
            x + w/2, y + h*0.28,
            subtitle,
            ha="center", va="center",
            fontsize=8
        )

def arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="-|>",
        mutation_scale=14,
        linewidth=1.2,
        color="#607D8B"
    ))

box(
    3.3, 8.8, 6.4, 0.8,
    "Patient-level healthcare data",
    "UCI diabetes; patient-disjoint development and held-out cohorts"
)

arrow(6.5, 8.8, 6.5, 8.35)

box(0.5, 7.45, 3.6, 0.85,
    "R1: Clinical",
    "Age and diagnosis families")

box(4.7, 7.45, 3.6, 0.85,
    "R2: Clinical + utilization",
    "Clinical features and prior-use counts")

box(8.9, 7.45, 3.6, 0.85,
    "R3: Clinical + utilization",
    "+ documentation observation indicators")

for cx in [2.3, 6.5, 10.7]:
    arrow(6.5, 8.35, cx, 8.30)

box(
    3.4, 6.15, 6.2, 0.8,
    "Candidate clustering solutions",
    "KMeans; five evaluated representation/K combinations"
)

for cx in [2.3, 6.5, 10.7]:
    arrow(cx, 7.45, 6.5, 6.95)

arrow(6.5, 6.15, 6.5, 5.75)

box(0.5, 4.85, 5.7, 0.8,
    "Internal quality and stability",
    "Silhouette; seed and subsample agreement")

box(6.8, 4.85, 5.7, 0.8,
    "Perturbation and representation tests",
    "Missingness; feature concentration; dominance")

arrow(6.5, 5.75, 3.35, 5.65)
arrow(6.5, 5.75, 9.65, 5.65)

box(
    3.1, 3.45, 6.8, 0.8,
    "Exploratory reliability assessment",
    "Multiple criteria; no clinical validation implied",
    face="#F2EDE2",
    edge="#B49B62"
)

arrow(3.35, 4.85, 6.5, 4.25)
arrow(9.65, 4.85, 6.5, 4.25)

box(
    0.8, 1.9, 5.1, 0.9,
    "Candidate retained",
    "If exploratory criteria satisfied",
    face="#E7F1EB",
    edge="#6C9B82"
)

box(
    7.1, 1.9, 5.1, 0.9,
    "Flag / abstain",
    "All five evaluated candidates flagged",
    face="#F6E9E8",
    edge="#B47773"
)

arrow(6.5, 3.45, 3.35, 2.8)
arrow(6.5, 3.45, 9.65, 2.8)

ax.text(
    6.5, 9.9,
    "Reliability-aware healthcare phenotyping framework",
    ha="center", va="top",
    fontsize=15, fontweight="bold"
)

ax.text(
    6.5, 0.9,
    "Framework illustrated using UCI diabetes data. "
    "Exploratory reliability criteria are not validated clinical safety thresholds.",
    ha="center", va="center",
    fontsize=8.5
)

png = FIGURES / "figure1_reliability_framework.png"
pdf = FIGURES / "figure1_reliability_framework.pdf"

fig.savefig(png, dpi=300, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")

plt.show()

print("Saved:", png)
print("Saved:", pdf)
print("STEP 31E: COMPLETE")
